# Fretwork Debug CAGED Voiced — Audio → Tab + Strict Conservative Repair

This notebook contains the GuitarSet parser, held-out train/validation/test split, fretboard assignment algorithms, Basic Pitch transcription, audio-derived key/chord context, and final end-to-end evaluation in one place. It does **not** import from any other project notebooks.

Prediction path: actual audio → Basic Pitch note events → detected key/chord context → `combined_all_tuned` fretboard assignment → predicted string/fret tab.

Ground truth path: GuitarSet `.jams` files are used only after prediction for scoring.


**Added in this version:** a stricter constrained repair layer. It avoids broad smoothing and only rewrites clearly suspicious string/fret choices: invalid positions, MIDI-position mismatches, duplicate chord strings, extremely wide chord spans, or isolated one-off position outliers.

The notebook keeps original methods and adds repaired variants so you can directly compare whether repair improves GuitarSet held-out metrics and ASCII tab output. It also prints measure-level before/after contexts for repaired notes so you can see why the repair fired.


In [1]:
from pathlib import Path
from itertools import product
from collections import defaultdict
import json
import math
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [2]:
# Mount Google Drive when running in Colab.
# This must run before any /content/drive/MyDrive/... paths are used.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except ModuleNotFoundError:
    print('Not running in Colab. Skipping Google Drive mount.')


Mounted at /content/drive
Google Drive mounted successfully.


## 1. Configuration

This notebook assumes the GuitarSet data is somewhere in your mounted Google Drive, ideally one of these:

```text
/content/drive/MyDrive/Capstone/FullGuitarSetData
/content/drive/MyDrive/FullGuitarSetData
```

Expected data structure:

```text
FullGuitarSetData/
├── JamsFiles/
└── AudioFiles/
```

The CSV outputs are saved to:

```text
/content/drive/MyDrive/Capstone/outputs/fretboard_playability/
```

If the notebook cannot find the data folder automatically, update `DATA_ROOT_CANDIDATES` in the next cell.


In [3]:
# -------------------------
# USER CONFIG
# -------------------------

# Where outputs should go in Google Drive.
# This creates: My Drive / Capstone / outputs / fretboard_playability
# In local/non-Colab execution, this path may be created locally, but in Colab it writes to Drive after mounting.
CAPSTONE_ROOT = Path('/content/drive/MyDrive/Capstone')
OUTPUT_DIR = CAPSTONE_ROOT / 'outputs' / 'fretboard_playability'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Candidate locations for the GuitarSet data folder.
# Update/add to this list if your FullGuitarSetData or GuitarSet folder is somewhere else.
# The notebook will choose the first candidate that actually contains .jams files.
DATA_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Capstone/FullGuitarSetData'),
    Path('/content/drive/MyDrive/FullGuitarSetData'),
    Path('/content/drive/MyDrive/Capstone/GuitarSet'),
    Path('/content/drive/MyDrive/GuitarSet'),
    Path('/content/drive/MyDrive/Capstone'),
    Path('/content/drive/MyDrive'),
    Path('/mnt/data/fretwork_repo/GuitarSet'),
    Path('/mnt/data/fretwork_repo'),
]

MAX_FRET = 24
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]  # E2, A2, D3, G3, B3, E4
STRING_NAMES = ['low_E', 'A', 'D', 'G', 'B', 'high_E']
ONSET_TOLERANCE_SECONDS = 0.035

COMFORTABLE_SPAN = 5
MAX_REACHABLE_SPAN = 7
LARGE_JUMP_THRESHOLD = 5
MAX_GROUP_CANDIDATES = 25  # optimization cap for chord candidate combinations, not a demo note limit

# Tuned combined-all settings.
# `combined_all_tuned` uses empirically learned GuitarSet position priors plus adjustable weights.
# Leave RUN_WEIGHT_TUNING = True for the fastest full run. Set True if you want to run the small
# preset search below before the full evaluation.
RUN_WEIGHT_TUNING = True
TUNING_RECORD_LIMIT = 24
TUNING_OBJECTIVE_LARGE_JUMP_PENALTY = 0.35
TUNING_OBJECTIVE_DUPLICATE_STRING_PENALTY = 0.50

# Held-out evaluation settings.
# These make `combined_all_tuned` valid: train builds the position prior,
# validation selects preset weights, and test is unseen data for final reporting.
USE_HELDOUT_SPLIT = True
SPLIT_SEED = 42
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15


print(f'OUTPUT_DIR: {OUTPUT_DIR.resolve()}')
print('OUTPUT_DIR exists:', OUTPUT_DIR.exists())
print('\nData root candidates visible to this runtime:')
for p in DATA_ROOT_CANDIDATES:
    print(f' - {p} | exists: {p.exists()}')

if Path('/content/drive/MyDrive').exists():
    print('\nTop-level MyDrive folders/files visible to Colab:')
    for p in list(Path('/content/drive/MyDrive').iterdir())[:25]:
        print(' -', p.name)


OUTPUT_DIR: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/fretboard_playability
OUTPUT_DIR exists: True

Data root candidates visible to this runtime:
 - /content/drive/MyDrive/Capstone/FullGuitarSetData | exists: True
 - /content/drive/MyDrive/FullGuitarSetData | exists: False
 - /content/drive/MyDrive/Capstone/GuitarSet | exists: True
 - /content/drive/MyDrive/GuitarSet | exists: False
 - /content/drive/MyDrive/Capstone | exists: True
 - /content/drive/MyDrive | exists: True
 - /mnt/data/fretwork_repo/GuitarSet | exists: False
 - /mnt/data/fretwork_repo | exists: False

Top-level MyDrive folders/files visible to Colab:
 - DATASCI 201 Belmont Report Assignment.gdoc
 - Critique a Design Pt 3.gdoc
 - Unit 08 Assignment.gdoc
 - The Revival of High School Shop Classes: A Hands-On Comeback Story.gslides
 - Unit 9 Assignment - Aaron Luong.mp4
 - Unit 9 Assignment - Slides & Notes.gdoc
 - Career Compass.gdoc
 - Career Compass.mp4
 - Career Compass 

## 2. Fretboard Layout and MIDI Lookups

In [4]:
def build_fretboard(open_string_midi=OPEN_STRING_MIDI, max_fret=MAX_FRET):
    rows = []
    for string_idx, open_midi in enumerate(open_string_midi):
        for fret in range(max_fret + 1):
            midi = open_midi + fret
            rows.append({
                'string': string_idx,
                'string_name': STRING_NAMES[string_idx],
                'fret': fret,
                'midi': midi,
                'pitch_class': midi % 12,
            })
    return pd.DataFrame(rows)

fretboard_df = build_fretboard()
MIDI_TO_POSITIONS = defaultdict(list)
for row in fretboard_df.to_dict('records'):
    MIDI_TO_POSITIONS[int(row['midi'])].append({
        'string': int(row['string']),
        'string_name': row['string_name'],
        'fret': int(row['fret']),
        'midi': int(row['midi']),
        'pitch_class': int(row['pitch_class']),
    })

def get_possible_positions(midi_note, max_fret=MAX_FRET):
    midi_note = int(round(midi_note))
    return [p for p in MIDI_TO_POSITIONS.get(midi_note, []) if 0 <= p['fret'] <= max_fret]

print('Fretboard rows:', len(fretboard_df))
display(fretboard_df.head(12))
print('Example positions for MIDI 64 / E4:')
display(pd.DataFrame(get_possible_positions(64)))


Fretboard rows: 150


,string,string_name,fret,midi,pitch_class
0,0,low_E,0,40,4
1,0,low_E,1,41,5
2,0,low_E,2,42,6
3,0,low_E,3,43,7
4,0,low_E,4,44,8
5,0,low_E,5,45,9
6,0,low_E,6,46,10
7,0,low_E,7,47,11
8,0,low_E,8,48,0
9,0,low_E,9,49,1


Example positions for MIDI 64 / E4:


,string,string_name,fret,midi,pitch_class
0,0,low_E,24,64,4
1,1,A,19,64,4
2,2,D,14,64,4
3,3,G,9,64,4
4,4,B,5,64,4
5,5,high_E,0,64,4


## 3. Scale, Key, and Diatonic Chord Knowledge

In [5]:
PITCH_CLASS_NAMES_SHARP = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
NOTE_TO_PC = {name: i for i, name in enumerate(PITCH_CLASS_NAMES_SHARP)}
NOTE_TO_PC.update({'Db': 1, 'Eb': 3, 'Gb': 6, 'Ab': 8, 'Bb': 10})
PC_TO_NOTE = {i: name for i, name in enumerate(PITCH_CLASS_NAMES_SHARP)}

MAJOR_STEPS = [2, 2, 1, 2, 2, 2, 1]
MINOR_STEPS = [2, 1, 2, 2, 1, 2, 2]
MAJOR_QUALITIES = ['maj', 'min', 'min', 'maj', 'maj', 'min', 'dim']
MINOR_QUALITIES = ['min', 'dim', 'maj', 'min', 'min', 'maj', 'maj']

def derive_scale(root_pc, mode='major'):
    steps = MAJOR_STEPS if mode == 'major' else MINOR_STEPS
    pcs = [root_pc]
    cur = root_pc
    for step in steps[:-1]:
        cur = (cur + step) % 12
        pcs.append(cur)
    return pcs

def build_key_database():
    rows = []
    for root_name, root_pc in NOTE_TO_PC.items():
        if 'b' in root_name:
            continue
        for mode in ['major', 'minor']:
            scale_pcs = derive_scale(root_pc, mode)
            qualities = MAJOR_QUALITIES if mode == 'major' else MINOR_QUALITIES
            chords = []
            for degree, (pc, qual) in enumerate(zip(scale_pcs, qualities), start=1):
                chords.append({
                    'degree': degree,
                    'root_pc': pc,
                    'root': PC_TO_NOTE[pc],
                    'quality': qual,
                    'symbol': f'{PC_TO_NOTE[pc]}:{qual}',
                })
            rows.append({
                'key': f'{root_name} {mode}',
                'root': root_name,
                'root_pc': root_pc,
                'mode': mode,
                'scale_pcs': scale_pcs,
                'scale_notes': [PC_TO_NOTE[pc] for pc in scale_pcs],
                'diatonic_chords': chords,
            })
    return pd.DataFrame(rows)

key_db = build_key_database()
display(key_db.head())
print('D major diatonic chords:')
d_major = key_db[key_db['key'] == 'D major'].iloc[0]
print([c['symbol'] for c in d_major['diatonic_chords']])


,key,root,root_pc,mode,scale_pcs,scale_notes,diatonic_chords
0,C major,C,0,major,"[0, 2, 4, 5, 7, 9, 11]","[C, D, E, F, G, A, B]","[{'degree': 1, 'root_pc': 0, 'root': 'C', 'qua..."
1,C minor,C,0,minor,"[0, 2, 3, 5, 7, 8, 10]","[C, D, D#, F, G, G#, A#]","[{'degree': 1, 'root_pc': 0, 'root': 'C', 'qua..."
2,C# major,C#,1,major,"[1, 3, 5, 6, 8, 10, 0]","[C#, D#, F, F#, G#, A#, C]","[{'degree': 1, 'root_pc': 1, 'root': 'C#', 'qu..."
3,C# minor,C#,1,minor,"[1, 3, 4, 6, 8, 9, 11]","[C#, D#, E, F#, G#, A, B]","[{'degree': 1, 'root_pc': 1, 'root': 'C#', 'qu..."
4,D major,D,2,major,"[2, 4, 6, 7, 9, 11, 1]","[D, E, F#, G, A, B, C#]","[{'degree': 1, 'root_pc': 2, 'root': 'D', 'qua..."


D major diatonic chords:
['D:maj', 'E:min', 'F#:min', 'G:maj', 'A:maj', 'B:min', 'C#:dim']


## 4. Chord Knowledge and Recognition

In [6]:
CHORD_INTERVALS = {
    'maj': [0, 4, 7],
    'min': [0, 3, 7],
    'dim': [0, 3, 6],
    'aug': [0, 4, 8],
    '7': [0, 4, 7, 10],
    'maj7': [0, 4, 7, 11],
    'min7': [0, 3, 7, 10],
    'm7': [0, 3, 7, 10],
    'sus4': [0, 5, 7],
    'sus2': [0, 2, 7],
    '5': [0, 7],
}
QUALITY_ALIASES = {'M': 'maj', 'major': 'maj', '': 'maj', 'm': 'min', 'minor': 'min', 'dom7': '7'}

def normalize_quality(q):
    if q is None:
        return 'maj'
    q = str(q).strip()
    return QUALITY_ALIASES.get(q, q)

def chord_tones(root_pc, quality='maj'):
    quality = normalize_quality(quality)
    intervals = CHORD_INTERVALS.get(quality, CHORD_INTERVALS['maj'])
    return sorted({(root_pc + i) % 12 for i in intervals})

def parse_chord_symbol(symbol):
    if symbol is None:
        return None
    s = str(symbol).strip()
    # Remove inversion/bass-note suffixes such as D:7/1 or C:maj/G before parsing quality.
    s = s.split('/')[0]
    if s in ['N', 'X', 'nan', 'None', '']:
        return None
    if ':' in s:
        root, qual = s.split(':', 1)
    else:
        m = re.match(r'^([A-G](?:#|b)?)(.*)$', s)
        if not m:
            return None
        root, qual = m.group(1), m.group(2)
    if root not in NOTE_TO_PC:
        return None
    qual = normalize_quality(qual)
    return {'root': root, 'root_pc': NOTE_TO_PC[root], 'quality': qual, 'tones': chord_tones(NOTE_TO_PC[root], qual)}

def recognize_chord_from_pitches(midi_pitches, allowed_qualities=('maj', 'min', 'dim', '7', 'maj7', 'min7')):
    pcs = sorted({int(round(m)) % 12 for m in midi_pitches})
    if not pcs:
        return None
    best = None
    for root_pc in range(12):
        for qual in allowed_qualities:
            tones = set(chord_tones(root_pc, qual))
            pcs_set = set(pcs)
            precision = len(pcs_set & tones) / max(len(pcs_set), 1)
            recall = len(pcs_set & tones) / max(len(tones), 1)
            score = 2 * precision * recall / (precision + recall + 1e-9)
            cand = {'symbol': f'{PC_TO_NOTE[root_pc]}:{qual}', 'root_pc': root_pc, 'quality': qual, 'tones': sorted(tones), 'score': score}
            if best is None or cand['score'] > best['score']:
                best = cand
    return best

print(parse_chord_symbol('D:maj'))
print(recognize_chord_from_pitches([62, 66, 69]))


{'root': 'D', 'root_pc': 2, 'quality': 'maj', 'tones': [2, 6, 9]}
{'symbol': 'D:maj', 'root_pc': 2, 'quality': 'maj', 'tones': [2, 6, 9], 'score': 0.9999999995}


## 5. GuitarSet JAMS Parsing

This parser avoids requiring the external `jams` package. It directly reads the JSON-like `.jams` files.

In [7]:
def find_jams_dir(data_root):
    """Return a directory containing .jams files under data_root, or None if not found."""
    candidates = [
        data_root / 'JamsFiles',
        data_root / 'Annotations',
        data_root / 'GuitarSet' / 'Annotations',
        data_root / 'FullGuitarSetData' / 'JamsFiles',
        data_root / 'FullGuitarSetData' / 'Annotations',
        data_root,
    ]
    for c in candidates:
        if c.exists() and list(c.glob('*.jams')):
            return c

    # Last-resort recursive search under this candidate.
    # Limit to the first match to avoid loading the full Drive tree unnecessarily.
    if data_root.exists():
        try:
            for match in data_root.rglob('*.jams'):
                return match.parent
        except Exception as e:
            print(f'Could not recursively search {data_root}: {e}')
    return None

def choose_data_root_and_jams_dir(candidates):
    checked = []
    for root in candidates:
        checked.append((root, root.exists()))
        if not root.exists():
            continue
        jams_dir = find_jams_dir(root)
        if jams_dir is not None:
            return root, jams_dir

    print('Could not find .jams files automatically.')
    print('Checked these DATA_ROOT_CANDIDATES:')
    for root, exists in checked:
        print(f' - {root} | exists: {exists}')
    raise FileNotFoundError(
        'Could not find any .jams files. Add the correct GuitarSet/FullGuitarSetData path to DATA_ROOT_CANDIDATES.'
    )

DATA_ROOT, JAMS_DIR = choose_data_root_and_jams_dir(DATA_ROOT_CANDIDATES)
JAMS_FILES = sorted(JAMS_DIR.glob('*.jams'))
print(f'DATA_ROOT selected: {DATA_ROOT}')
print(f'Found {len(JAMS_FILES)} JAMS files in {JAMS_DIR}')
print('\n'.join(p.name for p in JAMS_FILES[:10]))

def get_annotation_data(annotation):
    data = annotation.get('data', [])
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        keys = ['time', 'duration', 'value', 'confidence']
        n = len(data.get('time', []))
        return [{k: data.get(k, [None] * n)[i] for k in keys} for i in range(n)]
    return []

def parse_string_from_data_source(data_source):
    """GuitarSet stores each string as a separate note_midi annotation with data_source 0-5."""
    try:
        s = int(data_source)
        return s if 0 <= s <= 5 else None
    except Exception:
        return None

def parse_jams_file(path):
    with open(path, 'r') as f:
        jam = json.load(f)
    notes, chords, beats = [], [], []
    tempo, key = None, None
    for ann in jam.get('annotations', []):
        ns = ann.get('namespace')
        rows = get_annotation_data(ann)
        data_source = ann.get('annotation_metadata', {}).get('data_source', '')
        if ns == 'note_midi':
            inferred_string = parse_string_from_data_source(data_source)
            for r in rows:
                v = r.get('value')
                if isinstance(v, dict):
                    midi = v.get('midi_note') or v.get('note') or v.get('pitch')
                    string = v.get('string', inferred_string)
                    fret = v.get('fret')
                else:
                    midi = v
                    string = inferred_string
                    fret = None
                if midi is None:
                    continue
                midi_int = int(round(float(midi)))
                # GuitarSet note_midi annotations usually give string via annotation data_source.
                # If fret is not explicitly stored, derive it from MIDI pitch and the open string pitch.
                if string is not None and fret is None:
                    fret = midi_int - OPEN_STRING_MIDI[int(string)]
                notes.append({
                    'start': float(r.get('time', 0.0)),
                    'duration': float(r.get('duration', 0.0) or 0.0),
                    'midi': midi_int,
                    'pitch_class': midi_int % 12,
                    'true_string': None if string is None else int(string),
                    'true_fret': None if fret is None else int(round(float(fret))),
                    'source': data_source,
                })
        elif ns in ['chord', 'chord_harte']:
            for r in rows:
                start = float(r.get('time', 0.0))
                duration = float(r.get('duration', 0.0) or 0.0)
                chord_label = r.get('value')
                chords.append({
                    'start': start,
                    'duration': duration,
                    'end': start + duration,
                    'chord': chord_label,
                    'parsed': parse_chord_symbol(chord_label),
                })
        elif ns in ['beat', 'beat_position']:
            for r in rows:
                beats.append(float(r.get('time', 0.0)))
        elif ns == 'key_mode':
            if rows:
                key = rows[0].get('value')
        elif ns == 'tempo':
            if rows:
                tempo = rows[0].get('value')
    notes = sorted(notes, key=lambda x: (x['start'], x['midi']))
    chords = sorted(chords, key=lambda x: x['start'])
    return {'recording': path.stem, 'path': str(path), 'notes': notes, 'chords': chords, 'beats': beats, 'tempo': tempo, 'key': key}

records = [parse_jams_file(p) for p in JAMS_FILES]
print('Parsed records:', len(records))
if records:
    print('Example record:', records[0]['recording'])
    print('Notes:', len(records[0]['notes']), 'Chords:', len(records[0]['chords']), 'Key:', records[0]['key'])
    display(pd.DataFrame(records[0]['notes']).head())
else:
    raise ValueError('No records parsed. Check JAMS_FILES and DATA_ROOT_CANDIDATES.')


DATA_ROOT selected: /content/drive/MyDrive/Capstone/FullGuitarSetData
Found 360 JAMS files in /content/drive/MyDrive/Capstone/FullGuitarSetData/JamsFiles
00_BN1-129-Eb_comp.jams
00_BN1-129-Eb_solo.jams
00_BN1-147-Gb_comp.jams
00_BN1-147-Gb_solo.jams
00_BN2-131-B_comp.jams
00_BN2-131-B_solo.jams
00_BN2-166-Ab_comp.jams
00_BN2-166-Ab_solo.jams
00_BN3-119-G_comp.jams
00_BN3-119-G_solo.jams
Parsed records: 360
Example record: 00_BN1-129-Eb_comp
Notes: 133 Chords: 12 Key: Eb:major


,start,duration,midi,pitch_class,true_string,true_fret,source
0,0.048816,0.423764,51,3,1,6,1
1,0.049791,0.452789,65,5,4,6,4
2,0.052717,0.458594,62,2,3,7,3
3,0.519995,0.417959,51,3,1,6,1
4,0.722036,0.859138,58,10,2,8,2


In [8]:

# -------------------------
# Valid train/validation/test split by recording
# -------------------------
# Important: split by recording, not by individual note, so notes from the same performance
# do not leak across train/validation/test.

import random


def split_records_by_recording(records, train_frac=0.70, val_frac=0.15, test_frac=0.15, seed=42):
    if not np.isclose(train_frac + val_frac + test_frac, 1.0):
        raise ValueError('train_frac + val_frac + test_frac must sum to 1.0')

    rng = random.Random(seed)

    # Keep solo/comp proportions roughly stable across splits when possible.
    groups = {
        'solo': [r for r in records if r['recording'].endswith('_solo')],
        'comp': [r for r in records if r['recording'].endswith('_comp')],
        'other': [r for r in records if not (r['recording'].endswith('_solo') or r['recording'].endswith('_comp'))],
    }

    train, val, test = [], [], []
    for label, group in groups.items():
        group = list(group)
        rng.shuffle(group)
        n = len(group)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        # Make sure split sizes add exactly to n.
        n_train = min(n_train, n)
        n_val = min(n_val, n - n_train)
        train.extend(group[:n_train])
        val.extend(group[n_train:n_train + n_val])
        test.extend(group[n_train + n_val:])

    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)
    return train, val, test


TRAIN_RECORDS, VAL_RECORDS, TEST_RECORDS = split_records_by_recording(
    records,
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC,
    seed=SPLIT_SEED,
)

print('Held-out split by recording:')
print(f'  Train records: {len(TRAIN_RECORDS)}')
print(f'  Validation records: {len(VAL_RECORDS)}')
print(f'  Test records: {len(TEST_RECORDS)}')
print(f'  Total records: {len(TRAIN_RECORDS) + len(VAL_RECORDS) + len(TEST_RECORDS)}')

split_rows = []
for split_name, split_records in [('train', TRAIN_RECORDS), ('validation', VAL_RECORDS), ('test', TEST_RECORDS)]:
    for r in split_records:
        split_rows.append({
            'recording': r['recording'],
            'split': split_name,
            'is_solo': r['recording'].endswith('_solo'),
            'is_comp': r['recording'].endswith('_comp'),
            'n_notes': len(r.get('notes', [])),
            'n_chords': len(r.get('chords', [])),
        })

split_df = pd.DataFrame(split_rows)
split_path = OUTPUT_DIR / 'fretboard_train_val_test_split.csv'
split_df.to_csv(split_path, index=False)
print('Saved split file to:', split_path.resolve())
display(split_df.groupby(['split', 'is_solo', 'is_comp']).agg(recordings=('recording', 'nunique'), notes=('n_notes', 'sum')).reset_index())


Held-out split by recording:
  Train records: 252
  Validation records: 54
  Test records: 54
  Total records: 360
Saved split file to: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/fretboard_playability/fretboard_train_val_test_split.csv


,split,is_solo,is_comp,recordings,notes
0,test,False,True,27,7364
1,test,True,False,27,2843
2,train,False,True,126,31543
3,train,True,False,126,11572
4,validation,False,True,27,6708
5,validation,True,False,27,2446


## 6. Context Helpers: Key and Chord at Each Note

In [9]:
def infer_key_from_filename(recording_name):
    parts = recording_name.split('_')
    if len(parts) >= 2:
        middle = parts[1]
        key_guess = middle.split('-')[-1]
        if key_guess in NOTE_TO_PC:
            return f'{key_guess} major'
    return None

def get_key_info(key_label):
    if key_label is None:
        return None
    s = str(key_label).replace(':', ' ').strip()
    # Normalize GuitarSet-style labels such as D:major into D major.
    toks = s.split()
    if len(toks) == 1 and toks[0] in NOTE_TO_PC:
        s = f'{toks[0]} major'
    match = key_db[key_db['key'] == s]
    return match.iloc[0].to_dict() if len(match) else None

def chord_end_time(c):
    # Some parsed chord dictionaries have duration but not an explicit end time.
    # This helper keeps the rest of the notebook robust either way.
    start = float(c.get('start', 0.0))
    if c.get('end') is not None:
        return float(c['end'])
    return start + float(c.get('duration', 0.0) or 0.0)

def chord_at_time(chords, t):
    for c in chords:
        start = float(c.get('start', 0.0))
        end = chord_end_time(c)
        if start <= t < end:
            return c
    return None

def enrich_notes_with_context(record):
    key_label = record.get('key') or infer_key_from_filename(record['recording'])
    key_info = get_key_info(key_label)
    out = []
    for n in record['notes']:
        c = chord_at_time(record['chords'], n['start'])
        row = dict(n)
        row['key_label'] = key_label
        row['in_key'] = None if key_info is None else (n['pitch_class'] in set(key_info['scale_pcs']))
        row['chord_label'] = None if c is None else c['chord']
        parsed_chord = None if c is None else c.get('parsed') or parse_chord_symbol(c.get('chord'))
        row['in_chord'] = None if parsed_chord is None else (n['pitch_class'] in set(parsed_chord['tones']))
        out.append(row)
    return out

sample_context = pd.DataFrame(enrich_notes_with_context(records[0]))
display(sample_context.head())


,start,duration,midi,pitch_class,true_string,true_fret,source,key_label,in_key,chord_label,in_chord
0,0.048816,0.423764,51,3,1,6,1,Eb:major,None,D#:maj,True
1,0.049791,0.452789,65,5,4,6,4,Eb:major,None,D#:maj,False
2,0.052717,0.458594,62,2,3,7,3,Eb:major,None,D#:maj,False
3,0.519995,0.417959,51,3,1,6,1,Eb:major,None,D#:maj,True
4,0.722036,0.859138,58,10,2,8,2,Eb:major,None,D#:maj,True


## 7. Playability Rules and Scoring

In [10]:
def estimate_hand_position_from_frets(frets):
    fretted = [f for f in frets if f > 0]
    return 0 if not fretted else int(round(np.median(fretted)))

def group_span(frets):
    fretted = [f for f in frets if f > 0]
    return 0 if len(fretted) <= 1 else max(fretted) - min(fretted)

def awkward_fingering_penalty(position, hand_center):
    fret = position['fret']
    if fret == 0:
        return 0.0
    distance = abs(fret - hand_center)
    if distance <= 2:
        return 0.0
    if distance <= COMFORTABLE_SPAN:
        return 0.5 * (distance - 2)
    if distance <= MAX_REACHABLE_SPAN:
        return 2.0 + (distance - COMFORTABLE_SPAN)
    return 10.0 + 2.0 * (distance - MAX_REACHABLE_SPAN)

def group_playability_cost(group_positions):
    if not group_positions:
        return 0.0
    strings = [p['string'] for p in group_positions]
    frets = [p['fret'] for p in group_positions]
    fretted = [f for f in frets if f > 0]
    cost = 0.0
    if len(strings) != len(set(strings)):
        return float('inf')
    span = group_span(frets)
    if span > COMFORTABLE_SPAN:
        cost += 2.0 * (span - COMFORTABLE_SPAN)
    if span > MAX_REACHABLE_SPAN:
        cost += 25.0 * (span - MAX_REACHABLE_SPAN)
    if fretted and min(fretted) <= 2 and max(fretted) >= 9:
        cost += 8.0
    if len(strings) >= 2:
        string_span = max(strings) - min(strings)
        if string_span > 4 and len(strings) <= 3:
            cost += 1.5 * (string_span - 4)
    hand_center = estimate_hand_position_from_frets(frets)
    cost += sum(awkward_fingering_penalty(p, hand_center) for p in group_positions)
    if any(f == 0 for f in frets) and fretted and max(fretted) > 7:
        cost += 3.0
    return cost

def transition_cost(prev_group, curr_group):
    if prev_group is None or curr_group is None:
        return 0.0
    prev_frets = [p['fret'] for p in prev_group]
    curr_frets = [p['fret'] for p in curr_group]
    prev_strings = [p['string'] for p in prev_group]
    curr_strings = [p['string'] for p in curr_group]
    prev_center = estimate_hand_position_from_frets(prev_frets)
    curr_center = estimate_hand_position_from_frets(curr_frets)
    cost = 1.2 * abs(curr_center - prev_center) + 0.25 * abs(np.mean(curr_strings) - np.mean(prev_strings))
    if len(prev_group) == 1 and len(curr_group) == 1:
        pf, cf = prev_group[0]['fret'], curr_group[0]['fret']
        ps, cs = prev_group[0]['string'], curr_group[0]['string']
        cost += 0.8 * abs(cf - pf) + 0.35 * abs(cs - ps)
        if abs(cf - pf) > LARGE_JUMP_THRESHOLD:
            cost += 4.0 + abs(cf - pf) - LARGE_JUMP_THRESHOLD
        if cf == 0 and pf > 7:
            cost += 2.0
    return cost

def context_cost(group_notes, group_positions):
    cost = 0.0
    for n, p in zip(group_notes, group_positions):
        if n.get('in_chord') is False:
            cost += 0.15
        if n.get('in_key') is False:
            cost += 0.10
    return cost


## 8. Group Notes by Onset

In [11]:
def group_notes_by_onset(notes, tolerance=ONSET_TOLERANCE_SECONDS):
    if not notes:
        return []
    notes_sorted = sorted(notes, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))
    groups, current = [], [notes_sorted[0]]
    group_start = notes_sorted[0]['start']
    for n in notes_sorted[1:]:
        if abs(n['start'] - group_start) <= tolerance:
            current.append(n)
        else:
            groups.append(current)
            current = [n]
            group_start = n['start']
    groups.append(current)
    return groups

def enrich_candidate(candidate):
    positions = candidate['positions']
    frets = [p['fret'] for p in positions]
    strings = [p['string'] for p in positions]
    candidate['center'] = estimate_hand_position_from_frets(frets)
    candidate['avg_string'] = float(np.mean(strings)) if strings else 0.0
    candidate['is_single'] = len(positions) == 1
    candidate['single_fret'] = positions[0]['fret'] if len(positions) == 1 else np.nan
    candidate['single_string'] = positions[0]['string'] if len(positions) == 1 else np.nan
    return candidate

def candidate_groups_for_notes(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)
    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue
        base_cost = group_playability_cost(combo) + context_cost(group_notes, combo)
        if math.isfinite(base_cost):
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))
    if not candidates:
        for combo in product(*position_lists):
            combo = list(combo)
            base_cost = group_playability_cost(combo)
            if math.isinf(base_cost):
                base_cost = 1000.0
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))
    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]

sample_groups = group_notes_by_onset(enrich_notes_with_context(records[0]))
print('Number of onset groups:', len(sample_groups))
print('First group size:', len(sample_groups[0]))
print('First group candidates:', len(candidate_groups_for_notes(sample_groups[0])))


Number of onset groups: 76
First group size: 3
First group candidates: 25


## 9. Baseline Assignment Methods

In [12]:
def choose_lowest_fret(midi):
    pos = get_possible_positions(midi)
    return None if not pos else min(pos, key=lambda p: (p['fret'], p['string']))

def choose_highest_string(midi):
    pos = get_possible_positions(midi)
    return None if not pos else max(pos, key=lambda p: (p['string'], -p['fret']))

def assign_baseline_lowest_fret(notes):
    out = []
    for n in notes:
        p = choose_lowest_fret(n['midi'])
        row = dict(n)
        row.update({'pred_string': None if p is None else p['string'], 'pred_fret': None if p is None else p['fret'], 'method': 'lowest_fret'})
        out.append(row)
    return out

def assign_baseline_highest_string(notes):
    out = []
    for n in notes:
        p = choose_highest_string(n['midi'])
        row = dict(n)
        row.update({'pred_string': None if p is None else p['string'], 'pred_fret': None if p is None else p['fret'], 'method': 'highest_string'})
        out.append(row)
    return out

def assign_nearest_previous(notes):
    groups = group_notes_by_onset(notes)
    pred_rows, prev_group = [], None
    for g in groups:
        candidates = candidate_groups_for_notes(g)
        if not candidates:
            continue
        best = min(candidates, key=lambda c: c['base_cost'] + transition_cost(prev_group, c['positions']))
        prev_group = best['positions']
        for n, p in zip(g, best['positions']):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'nearest_previous'})
            pred_rows.append(row)
    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))


## 10. Viterbi-Style Playability Assignment

In [13]:
def transition_cost_matrix(prev_cands, curr_cands):
    prev_center = np.array([c['center'] for c in prev_cands], dtype=float)
    curr_center = np.array([c['center'] for c in curr_cands], dtype=float)
    prev_str = np.array([c['avg_string'] for c in prev_cands], dtype=float)
    curr_str = np.array([c['avg_string'] for c in curr_cands], dtype=float)
    mat = 1.2 * np.abs(prev_center[:, None] - curr_center[None, :])
    mat += 0.25 * np.abs(prev_str[:, None] - curr_str[None, :])

    prev_single = np.array([c['is_single'] for c in prev_cands], dtype=bool)
    curr_single = np.array([c['is_single'] for c in curr_cands], dtype=bool)
    single_mask = prev_single[:, None] & curr_single[None, :]
    if single_mask.any():
        pf = np.array([c['single_fret'] for c in prev_cands], dtype=float)[:, None]
        cf = np.array([c['single_fret'] for c in curr_cands], dtype=float)[None, :]
        ps = np.array([c['single_string'] for c in prev_cands], dtype=float)[:, None]
        cs = np.array([c['single_string'] for c in curr_cands], dtype=float)[None, :]
        fret_diff = np.abs(cf - pf)
        string_diff = np.abs(cs - ps)
        extra = 0.8 * fret_diff + 0.35 * string_diff
        extra += np.where(fret_diff > LARGE_JUMP_THRESHOLD, 4.0 + fret_diff - LARGE_JUMP_THRESHOLD, 0.0)
        extra += np.where((cf == 0) & (pf > 7), 2.0, 0.0)
        mat += np.where(single_mask, extra, 0.0)
    return mat

def assign_viterbi_playability(notes):
    groups = group_notes_by_onset(notes)
    all_candidates = [candidate_groups_for_notes(g) for g in groups]
    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]

    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = transition_cost_matrix(prev_cands, curr_cands)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'viterbi_playability'})
            pred_rows.append(row)
    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))


## 10b. Original Teammate Algorithm + Combined-All Method

This section adds the original teammate logic into the same GuitarSet evaluation loop.

Methods added:

- `old_music_theory_greedy`: reproduces the original greedy music-theory-aware assignment style. It scores each valid position using key alignment, chord-tone membership, open-string bonus, fret-region comfort, and continuity from the previous note.
- `combined_all`: uses the new Viterbi/global optimization framework, but adds the original music-theory score as an additional candidate cost term on top of playability, span, open-string, chord/key, and transition rules.

This allows an apples-to-apples table comparing old/simple methods, the new playability method, and a combined method across the same GuitarSet records.

In [14]:

# -----------------------------------------------------------------------------
# Original teammate algorithm adapted for this notebook's data structures
# -----------------------------------------------------------------------------

OLD_THEORY_WEIGHTS = {
    'key_alignment': 1.0,
    'chord_tone': 2.0,
    'open_string_bonus': 1.0,
    'low_position_bonus': 0.5,
    'middle_neck_bonus': 0.3,
    'position_continuity': 0.5,
    'continuity_cap': 5.0,
}


def old_position_score(midi, position, note_row=None, previous_position=None, weights=None):
    """Higher-is-better score from the original music-theory-aware prototype.

    This adapts the old notebook's `score_position()` logic to the richer rows in this
    notebook. The score uses key/chord flags already computed by `enrich_notes_with_context`.
    """
    if weights is None:
        weights = OLD_THEORY_WEIGHTS

    fret = position['fret']
    score = 0.0

    if note_row is not None and note_row.get('in_key') is True:
        score += weights['key_alignment']

    if note_row is not None and note_row.get('in_chord') is True:
        score += weights['chord_tone']

    if fret == 0:
        score += weights['open_string_bonus']
    elif fret <= 3:
        score += weights['low_position_bonus']
    elif 4 <= fret <= 12:
        score += weights['middle_neck_bonus']

    if previous_position is not None:
        prev_fret = previous_position['fret']
        if prev_fret > 0 and fret > 0:
            fret_distance = min(abs(fret - prev_fret), weights['continuity_cap'])
            score -= weights['position_continuity'] * (fret_distance ** 0.5)

    return float(score)


def assign_old_music_theory_greedy(notes):
    """Original teammate music-theory-aware assignment, evaluated over GuitarSet.

    Greedy per-note method:
    - enumerate valid positions for each MIDI note
    - score each position using old key/chord/comfort/continuity rules
    - choose the best local position

    For simultaneous notes, this remains per-note and can therefore reveal duplicate-string
    violations, which is useful when comparing old vs. new playability rules.
    """
    pred_rows = []
    previous_position = None

    for n in sorted(notes, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi'])):
        positions = get_possible_positions(n['midi'])
        row = dict(n)

        if not positions:
            row.update({'pred_string': None, 'pred_fret': None, 'method': 'old_music_theory_greedy'})
            pred_rows.append(row)
            continue

        best = max(
            positions,
            key=lambda p: old_position_score(n['midi'], p, note_row=n, previous_position=previous_position)
        )
        row.update({'pred_string': best['string'], 'pred_fret': best['fret'], 'method': 'old_music_theory_greedy'})
        pred_rows.append(row)
        previous_position = best

    return pred_rows


# -----------------------------------------------------------------------------
# Original/simple Viterbi without the new playability/context rules
# -----------------------------------------------------------------------------

def original_candidate_groups_for_notes(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    """Simple original-style candidate groups.

    This uses valid fretboard positions and a small low-fret preference, but does not use
    the new playability span penalties, awkward fingering penalties, open-string context,
    or chord/key context costs.
    """
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)

    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)

        # Keep physically impossible chord shapes out, but otherwise keep this simple.
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue

        frets = [p['fret'] for p in combo]
        base_cost = 0.05 * float(np.mean(frets)) + 0.05 * float(np.std(frets))
        candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))

    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]


def original_transition_cost_matrix(prev_cands, curr_cands):
    """Movement-only transition cost for the simple/original Viterbi method."""
    prev_center = np.array([c['center'] for c in prev_cands], dtype=float)
    curr_center = np.array([c['center'] for c in curr_cands], dtype=float)
    prev_str = np.array([c['avg_string'] for c in prev_cands], dtype=float)
    curr_str = np.array([c['avg_string'] for c in curr_cands], dtype=float)

    mat = np.abs(prev_center[:, None] - curr_center[None, :])
    mat += 0.25 * np.abs(prev_str[:, None] - curr_str[None, :])
    return mat


def assign_viterbi_original(notes):
    """Simple/original Viterbi assignment for comparison with new playability Viterbi."""
    groups = group_notes_by_onset(notes)
    all_candidates = [original_candidate_groups_for_notes(g) for g in groups]
    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]

    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = original_transition_cost_matrix(prev_cands, curr_cands)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'viterbi_original'})
            pred_rows.append(row)

    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))


# -----------------------------------------------------------------------------
# Combined-all method: original theory score + new playability rules + Viterbi
# -----------------------------------------------------------------------------

def old_theory_group_cost(group_notes, group_positions):
    """Convert the old higher-is-better music theory score into a lower-is-better cost."""
    if not group_notes or not group_positions:
        return 0.0
    scores = []
    for n, p in zip(group_notes, group_positions):
        scores.append(old_position_score(n['midi'], p, note_row=n, previous_position=None))
    # Negative because our Viterbi minimizes cost. Scale modestly so it helps but does not dominate playability.
    return -0.35 * float(np.mean(scores))


def candidate_groups_combined_all(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    """Candidate generator combining all available signals.

    Includes:
    - valid string/fret lookup
    - duplicate-string constraint for chords
    - new playability span/stretch/open-string rules
    - new key/chord context penalties
    - old teammate music-theory score as a bonus
    """
    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)

    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue

        base_cost = group_playability_cost(combo) + context_cost(group_notes, combo) + old_theory_group_cost(group_notes, combo)
        if math.isfinite(base_cost):
            candidates.append(enrich_candidate({'positions': combo, 'base_cost': base_cost}))

    if not candidates:
        return candidate_groups_for_notes(group_notes, max_candidates=max_candidates)

    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]


def assign_combined_all(notes):
    """Full combined method: original theory + new playability + Viterbi sequence optimization."""
    groups = group_notes_by_onset(notes)
    all_candidates = [candidate_groups_combined_all(g) for g in groups]
    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]

    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = transition_cost_matrix(prev_cands, curr_cands)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': 'combined_all'})
            pred_rows.append(row)

    return sorted(pred_rows, key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi']))


# Quick smoke test on the first record.
smoke_notes = enrich_notes_with_context(records[0])[:50]
for name, fn in {
    'old_music_theory_greedy': assign_old_music_theory_greedy,
    'viterbi_original': assign_viterbi_original,
    'combined_all': assign_combined_all,
}.items():
    smoke_pred = fn(smoke_notes)
    print(f'{name}: produced {len(smoke_pred)} predictions')


old_music_theory_greedy: produced 50 predictions
viterbi_original: produced 50 predictions
combined_all: produced 50 predictions


## 10c. Combined-All Tuned Method

This adds a stronger `combined_all_tuned` method on top of `combined_all`.

New pieces:
- empirical GuitarSet position prior: `P(string, fret | midi)`
- configurable cost weights
- optional lightweight preset tuning
- Viterbi sequence optimization using the tuned costs

The position prior is the main data-driven addition. It learns which string/fret positions GuitarSet tends to use for each MIDI note, then gives lower cost to more common positions.

In [15]:

# -----------------------------------------------------------------------------
# Combined-all tuned method: empirical position priors + tuned weights + Viterbi
# -----------------------------------------------------------------------------

def build_position_prior(records, alpha=0.50):
    """Build an empirical prior over guitar positions: P(string, fret | midi).

    This is a data-driven guitaristic prior learned from GuitarSet annotations. For each
    MIDI note, it estimates how often each valid string/fret position is used in the
    annotations. It returns normalized costs where the most common position for each MIDI
    note has cost 0 and less common positions have positive cost.

    This notebook builds the prior from TRAIN_RECORDS only, then evaluates on held-out TEST_RECORDS. This avoids leakage from the test set into the learned position prior.
    """
    counts = {}
    for rec in records:
        for n in rec.get('notes', []):
            midi = n.get('midi')
            s = n.get('true_string')
            f = n.get('true_fret')
            if midi is None or s is None or f is None:
                continue
            try:
                midi = int(midi)
                s = int(s)
                f = int(f)
            except Exception:
                continue
            if not (0 <= s < len(OPEN_STRING_MIDI) and 0 <= f <= MAX_FRET):
                continue
            # Keep only physically valid ground-truth positions.
            if OPEN_STRING_MIDI[s] + f != midi:
                continue
            counts[(midi, s, f)] = counts.get((midi, s, f), 0) + 1

    prior_costs = {}
    prior_probs = {}

    for midi in range(min(MIDI_TO_POSITIONS.keys()), max(MIDI_TO_POSITIONS.keys()) + 1):
        positions = get_possible_positions(midi)
        if not positions:
            continue

        total = sum(counts.get((midi, p['string'], p['fret']), 0) for p in positions)
        denom = total + alpha * len(positions)

        raw_costs = []
        for p in positions:
            prob = (counts.get((midi, p['string'], p['fret']), 0) + alpha) / denom
            cost = -math.log(prob)
            raw_costs.append(cost)
            prior_probs[(midi, p['string'], p['fret'])] = prob

        # Normalize so the best empirical position for a MIDI note has 0 cost.
        min_cost = min(raw_costs)
        for p, cost in zip(positions, raw_costs):
            prior_costs[(midi, p['string'], p['fret'])] = cost - min_cost

    return prior_costs, prior_probs


PRIOR_SOURCE_RECORDS = TRAIN_RECORDS if USE_HELDOUT_SPLIT else records
POSITION_PRIOR_COSTS, POSITION_PRIOR_PROBS = build_position_prior(PRIOR_SOURCE_RECORDS)
print(f'Built empirical position prior from {len(PRIOR_SOURCE_RECORDS)} training records for {len(POSITION_PRIOR_COSTS)} MIDI/string/fret candidates.')


def position_prior_cost(midi, position):
    """Lower cost = position is more common for this MIDI note in GuitarSet."""
    key = (int(midi), int(position['string']), int(position['fret']))
    return float(POSITION_PRIOR_COSTS.get(key, 0.75))


DEFAULT_TUNED_WEIGHTS = {
    # Candidate/base costs
    'playability': 0.70,
    'context': 0.35,
    'old_theory': 0.45,
    'position_prior': 1.15,

    # Transition costs
    'hand_shift': 1.05,
    'string_shift': 0.22,
    'single_fret_shift': 0.65,
    'single_string_shift': 0.30,
    'large_jump_extra': 4.50,
    'open_after_high_extra': 2.25,

    # Extra group-shape preference
    'group_span_extra': 0.15,
}


def candidate_groups_combined_all_tuned(group_notes, weights=None, max_candidates=MAX_GROUP_CANDIDATES):
    """Candidate generator for the tuned combined-all method.

    It combines:
    - valid string/fret lookup
    - duplicate-string constraint for chords
    - playability rules
    - key/chord context
    - old teammate theory score
    - empirical GuitarSet position prior
    """
    if weights is None:
        weights = DEFAULT_TUNED_WEIGHTS

    position_lists = []
    for n in group_notes:
        pos = get_possible_positions(n['midi'])
        if not pos:
            return []
        position_lists.append(pos)

    candidates = []
    for combo in product(*position_lists):
        combo = list(combo)

        # Enforce physical chord feasibility.
        if len(combo) > 1 and len({p['string'] for p in combo}) != len(combo):
            continue

        play_cost = group_playability_cost(combo)
        if not math.isfinite(play_cost):
            continue

        ctx_cost = context_cost(group_notes, combo)
        old_cost = old_theory_group_cost(group_notes, combo)  # negative is good
        prior_cost = float(np.mean([position_prior_cost(n['midi'], p) for n, p in zip(group_notes, combo)]))

        frets = [p['fret'] for p in combo]
        span_extra = group_span(frets)

        base_cost = (
            weights['playability'] * play_cost
            + weights['context'] * ctx_cost
            + weights['old_theory'] * old_cost
            + weights['position_prior'] * prior_cost
            + weights['group_span_extra'] * span_extra
        )

        if math.isfinite(base_cost):
            cand = enrich_candidate({
                'positions': combo,
                'base_cost': float(base_cost),
                'playability_cost': float(play_cost),
                'context_cost': float(ctx_cost),
                'old_theory_cost': float(old_cost),
                'position_prior_cost': float(prior_cost),
            })
            candidates.append(cand)

    if not candidates:
        return candidate_groups_combined_all(group_notes, max_candidates=max_candidates)

    return sorted(candidates, key=lambda c: c['base_cost'])[:max_candidates]


def tuned_transition_cost_matrix(prev_cands, curr_cands, weights=None):
    """Transition matrix for tuned combined-all.

    Similar to the playability Viterbi transition matrix, but all major costs are
    parameterized so they can be tuned.
    """
    if weights is None:
        weights = DEFAULT_TUNED_WEIGHTS

    prev_center = np.array([c['center'] for c in prev_cands], dtype=float)
    curr_center = np.array([c['center'] for c in curr_cands], dtype=float)
    prev_str = np.array([c['avg_string'] for c in prev_cands], dtype=float)
    curr_str = np.array([c['avg_string'] for c in curr_cands], dtype=float)

    mat = weights['hand_shift'] * np.abs(prev_center[:, None] - curr_center[None, :])
    mat += weights['string_shift'] * np.abs(prev_str[:, None] - curr_str[None, :])

    prev_single = np.array([c['is_single'] for c in prev_cands], dtype=bool)
    curr_single = np.array([c['is_single'] for c in curr_cands], dtype=bool)
    single_mask = prev_single[:, None] & curr_single[None, :]

    if single_mask.any():
        pf = np.array([c['single_fret'] for c in prev_cands], dtype=float)[:, None]
        cf = np.array([c['single_fret'] for c in curr_cands], dtype=float)[None, :]
        ps = np.array([c['single_string'] for c in prev_cands], dtype=float)[:, None]
        cs = np.array([c['single_string'] for c in curr_cands], dtype=float)[None, :]

        fret_diff = np.abs(cf - pf)
        string_diff = np.abs(cs - ps)
        extra = weights['single_fret_shift'] * fret_diff
        extra += weights['single_string_shift'] * string_diff
        extra += np.where(
            fret_diff > LARGE_JUMP_THRESHOLD,
            weights['large_jump_extra'] + fret_diff - LARGE_JUMP_THRESHOLD,
            0.0
        )
        extra += np.where((cf == 0) & (pf > 7), weights['open_after_high_extra'], 0.0)
        mat += np.where(single_mask, extra, 0.0)

    return mat


def assign_combined_all_tuned_with_weights(notes, weights=None, method_name='combined_all_tuned'):
    """Tuned combined-all assignment with caller-provided weights."""
    if weights is None:
        weights = DEFAULT_TUNED_WEIGHTS

    groups = group_notes_by_onset(notes)
    all_candidates = [candidate_groups_combined_all_tuned(g, weights=weights) for g in groups]

    if any(len(cands) == 0 for cands in all_candidates):
        raise ValueError('At least one group has no valid candidates.')

    dp = [np.array([c['base_cost'] for c in all_candidates[0]], dtype=float)]
    backptr = [np.full(len(dp[0]), -1, dtype=int)]

    for i in range(1, len(groups)):
        prev_cands, curr_cands = all_candidates[i - 1], all_candidates[i]
        trans = tuned_transition_cost_matrix(prev_cands, curr_cands, weights=weights)
        curr_base = np.array([c['base_cost'] for c in curr_cands], dtype=float)
        scores = dp[i - 1][:, None] + trans + curr_base[None, :]
        curr_back = np.argmin(scores, axis=0).astype(int)
        curr_costs = scores[curr_back, np.arange(scores.shape[1])]
        dp.append(curr_costs)
        backptr.append(curr_back)

    idx = int(np.argmin(dp[-1]))
    chosen_indices = [idx]
    for i in range(len(groups) - 1, 0, -1):
        idx = int(backptr[i][idx])
        chosen_indices.append(idx)
    chosen_indices = list(reversed(chosen_indices))

    pred_rows = []
    for g, cands, ci in zip(groups, all_candidates, chosen_indices):
        positions = cands[ci]['positions']
        for n, p in zip(g, positions):
            row = dict(n)
            row.update({'pred_string': p['string'], 'pred_fret': p['fret'], 'method': method_name})
            pred_rows.append(row)

    return sorted(
        pred_rows,
        key=lambda x: (x['start'], x.get('true_string') if x.get('true_string') is not None else 99, x['midi'])
    )


def assign_combined_all_tuned(notes):
    """Public method used in the full evaluation loop."""
    return assign_combined_all_tuned_with_weights(notes, weights=DEFAULT_TUNED_WEIGHTS, method_name='combined_all_tuned')


# Optional lightweight preset search. This is intentionally small so it can run in Colab.
# It updates DEFAULT_TUNED_WEIGHTS if RUN_WEIGHT_TUNING = True.
TUNED_WEIGHT_PRESETS = [
    {
        **DEFAULT_TUNED_WEIGHTS,
        'playability': 0.70,
        'old_theory': 0.45,
        'position_prior': 1.15,
        'single_fret_shift': 0.65,
    },
    {
        **DEFAULT_TUNED_WEIGHTS,
        'playability': 0.60,
        'old_theory': 0.35,
        'position_prior': 1.40,
        'single_fret_shift': 0.55,
    },
    {
        **DEFAULT_TUNED_WEIGHTS,
        'playability': 0.85,
        'old_theory': 0.30,
        'position_prior': 1.05,
        'single_fret_shift': 0.80,
    },
    {
        **DEFAULT_TUNED_WEIGHTS,
        'playability': 0.55,
        'old_theory': 0.55,
        'position_prior': 1.25,
        'single_fret_shift': 0.60,
    },
]


def tuning_objective(metrics):
    """Higher is better: accuracy with penalties for visibly bad playability."""
    return (
        float(metrics.get('exact_position_acc', 0.0))
        - TUNING_OBJECTIVE_LARGE_JUMP_PENALTY * float(metrics.get('large_jump_rate', 0.0))
        - TUNING_OBJECTIVE_DUPLICATE_STRING_PENALTY * float(metrics.get('duplicate_string_violation_rate', 0.0))
        - 0.05 * float(metrics.get('avg_fret_error', 0.0))
    )


def evaluate_weight_preset(records_subset, weights):
    rows = []
    for rec in records_subset:
        notes = enrich_notes_with_context(rec)
        notes = [
            n for n in notes
            if n.get('true_string') is not None
            and n.get('true_fret') is not None
            and 0 <= n['true_fret'] <= MAX_FRET
        ]
        if not notes:
            continue
        pred = assign_combined_all_tuned_with_weights(notes, weights=weights, method_name='combined_all_tuned_candidate')
        metrics, _ = evaluate_predictions(pred)
        rows.append(metrics)
    if not rows:
        return {'objective': -np.inf}
    df = pd.DataFrame(rows)
    agg = {
        'exact_position_acc': df['exact_position_acc'].mean(),
        'avg_fret_error': df['avg_fret_error'].mean(),
        'avg_fret_jump': df['avg_fret_jump'].mean(),
        'large_jump_rate': df['large_jump_rate'].mean(),
        'duplicate_string_violation_rate': df['duplicate_string_violation_rate'].mean(),
    }
    agg['objective'] = tuning_objective(agg)
    return agg


print('Tuned combined-all functions defined. Weight tuning will run after evaluation helpers are defined.')


Built empirical position prior from 252 training records for 150 MIDI/string/fret candidates.
Tuned combined-all functions defined. Weight tuning will run after evaluation helpers are defined.


## 11. Evaluation Metrics

In [16]:
def add_prediction_diagnostics(df):
    df = df.copy()
    df['pred_midi'] = [OPEN_STRING_MIDI[int(s)] + int(f) if pd.notna(s) and pd.notna(f) else np.nan for s, f in zip(df['pred_string'], df['pred_fret'])]
    df['correct_pitch_from_tab'] = df['pred_midi'] == df['midi']
    df['valid_position'] = df.apply(lambda r: pd.notna(r['pred_string']) and pd.notna(r['pred_fret']) and 0 <= int(r['pred_string']) <= 5 and 0 <= int(r['pred_fret']) <= MAX_FRET, axis=1)
    df['exact_position_correct'] = (df['pred_string'] == df['true_string']) & (df['pred_fret'] == df['true_fret'])
    df['string_correct'] = df['pred_string'] == df['true_string']
    df['fret_correct'] = df['pred_fret'] == df['true_fret']
    df['fret_error'] = (df['pred_fret'] - df['true_fret']).abs()
    df['string_error'] = (df['pred_string'] - df['true_string']).abs()
    return df

def duplicate_string_violation_rate(df, onset_tolerance=ONSET_TOLERANCE_SECONDS):
    if df.empty:
        return np.nan
    groups = group_notes_by_onset(df.sort_values(['start', 'midi']).to_dict('records'), tolerance=onset_tolerance)
    violations, total_chord_groups = 0, 0
    for g in groups:
        if len(g) <= 1:
            continue
        total_chord_groups += 1
        strings = [x.get('pred_string') for x in g if pd.notna(x.get('pred_string'))]
        if len(strings) != len(set(strings)):
            violations += 1
    return violations / total_chord_groups if total_chord_groups else 0.0

def average_group_span(df):
    if df.empty:
        return np.nan
    groups = group_notes_by_onset(df.sort_values(['start', 'midi']).to_dict('records'))
    spans = []
    for g in groups:
        frets = [int(x['pred_fret']) for x in g if pd.notna(x.get('pred_fret'))]
        if frets:
            spans.append(group_span(frets))
    return float(np.mean(spans)) if spans else np.nan

def movement_metrics(df):
    groups = group_notes_by_onset(df.sort_values(['start', 'midi']).to_dict('records'))
    centers, avg_strings = [], []
    for g in groups:
        frets = [int(x['pred_fret']) for x in g if pd.notna(x.get('pred_fret'))]
        strings = [int(x['pred_string']) for x in g if pd.notna(x.get('pred_string'))]
        if frets and strings:
            centers.append(estimate_hand_position_from_frets(frets))
            avg_strings.append(float(np.mean(strings)))
    if len(centers) <= 1:
        return {'avg_fret_jump': 0.0, 'avg_string_jump': 0.0, 'large_jump_rate': 0.0, 'large_jump_count': 0}
    fret_jumps = np.abs(np.diff(centers))
    string_jumps = np.abs(np.diff(avg_strings))
    large = fret_jumps > LARGE_JUMP_THRESHOLD
    return {'avg_fret_jump': float(np.mean(fret_jumps)), 'avg_string_jump': float(np.mean(string_jumps)), 'large_jump_rate': float(np.mean(large)), 'large_jump_count': int(np.sum(large))}

def evaluate_predictions(pred_rows):
    df = pd.DataFrame(pred_rows)
    if df.empty:
        return {}, df
    df = add_prediction_diagnostics(df)
    mv = movement_metrics(df)
    metrics = {
        'n_notes': len(df),
        'exact_position_acc': float(df['exact_position_correct'].mean()),
        'string_acc': float(df['string_correct'].mean()),
        'fret_acc': float(df['fret_correct'].mean()),
        'avg_fret_error': float(df['fret_error'].mean()),
        'avg_string_error': float(df['string_error'].mean()),
        'correct_pitch_from_tab_rate': float(df['correct_pitch_from_tab'].mean()),
        'valid_position_rate': float(df['valid_position'].mean()),
        'duplicate_string_violation_rate': float(duplicate_string_violation_rate(df)),
        'avg_group_span': float(average_group_span(df)),
        **mv,
    }
    return metrics, df


In [17]:

# -------------------------
# Tune combined_all_tuned on validation records only
# -------------------------
# This is what makes the tuned method a valid held-out evaluation:
# - TRAIN_RECORDS builds the position prior
# - VAL_RECORDS selects the best weight preset
# - TEST_RECORDS is used for final metrics only


def evaluate_weight_preset(records_subset, weights):
    rows = []
    for rec in records_subset:
        notes = enrich_notes_with_context(rec)
        notes = [
            n for n in notes
            if n.get('true_string') is not None
            and n.get('true_fret') is not None
            and 0 <= n['true_fret'] <= MAX_FRET
        ]
        if not notes:
            continue
        pred = assign_combined_all_tuned_with_weights(
            notes,
            weights=weights,
            method_name='combined_all_tuned_candidate'
        )
        metrics, _ = evaluate_predictions(pred)
        rows.append(metrics)
    if not rows:
        return {'objective': -np.inf}
    df = pd.DataFrame(rows)
    agg = {
        'exact_position_acc': df['exact_position_acc'].mean(),
        'avg_fret_error': df['avg_fret_error'].mean(),
        'avg_fret_jump': df['avg_fret_jump'].mean(),
        'large_jump_rate': df['large_jump_rate'].mean(),
        'duplicate_string_violation_rate': df['duplicate_string_violation_rate'].mean(),
    }
    agg['objective'] = tuning_objective(agg)
    return agg


if USE_HELDOUT_SPLIT:
    tuning_pool = VAL_RECORDS
    tuning_label = 'validation'
else:
    tuning_pool = records[:min(TUNING_RECORD_LIMIT, len(records))]
    tuning_label = 'exploratory_subset'

if RUN_WEIGHT_TUNING:
    print(f'Running lightweight tuning search over preset weights on {tuning_label} records...')
    tuning_records = tuning_pool[:min(TUNING_RECORD_LIMIT, len(tuning_pool))]
    tuning_rows = []
    for i, preset in enumerate(TUNED_WEIGHT_PRESETS):
        result = evaluate_weight_preset(tuning_records, preset)
        result['preset_id'] = i
        result['n_tuning_records'] = len(tuning_records)
        tuning_rows.append(result)

    tuning_df = pd.DataFrame(tuning_rows).sort_values('objective', ascending=False)
    tuning_path = OUTPUT_DIR / 'fretboard_tuning_results_validation.csv'
    tuning_df.to_csv(tuning_path, index=False)
    display(tuning_df)
    best_id = int(tuning_df.iloc[0]['preset_id'])
    DEFAULT_TUNED_WEIGHTS.update(TUNED_WEIGHT_PRESETS[best_id])
    print('Selected tuned preset:', best_id)
    print('Selected weights:', DEFAULT_TUNED_WEIGHTS)
    print('Saved tuning results to:', tuning_path.resolve())
else:
    print('RUN_WEIGHT_TUNING is False. Using default tuned weights:')
    print(DEFAULT_TUNED_WEIGHTS)

# Smoke test for tuned method after tuning has selected weights.
smoke_records = VAL_RECORDS if USE_HELDOUT_SPLIT and VAL_RECORDS else records
if smoke_records:
    tuned_smoke = assign_combined_all_tuned(enrich_notes_with_context(smoke_records[0])[:50])
    print(f'combined_all_tuned smoke test: produced {len(tuned_smoke)} predictions')


Running lightweight tuning search over preset weights on validation records...


,exact_position_acc,avg_fret_error,avg_fret_jump,large_jump_rate,duplicate_string_violation_rate,objective,preset_id,n_tuning_records
1,0.717002,1.353353,1.029694,0.000843,0.0,0.649039,1,24
0,0.714173,1.369699,1.019322,0.000843,0.0,0.645393,0,24
3,0.710572,1.384967,1.019380,0.000843,0.0,0.641028,3,24
2,0.699138,1.433684,1.012600,0.000843,0.0,0.627159,2,24


Selected tuned preset: 1
Selected weights: {'playability': 0.6, 'context': 0.35, 'old_theory': 0.35, 'position_prior': 1.4, 'hand_shift': 1.05, 'string_shift': 0.22, 'single_fret_shift': 0.55, 'single_string_shift': 0.3, 'large_jump_extra': 4.5, 'open_after_high_extra': 2.25, 'group_span_extra': 0.15}
Saved tuning results to: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/fretboard_playability/fretboard_tuning_results_validation.csv
combined_all_tuned smoke test: produced 50 predictions



## Audio Evaluation Setup

Run this after the held-out split and tuning cells above. The variables `TRAIN_RECORDS`, `VAL_RECORDS`, `TEST_RECORDS`, `POSITION_PRIOR_COSTS`, and `DEFAULT_TUNED_WEIGHTS` should already exist.

The final evaluation below uses only `TEST_RECORDS` when `USE_HELDOUT_SPLIT=True`.


In [18]:
# ============================================================
# FIX-ALL Basic Pitch install/import cell for Colab Python 3.12
# Run this ONCE after Runtime -> Restart runtime
# ============================================================

import sys
import subprocess
import importlib
import pkgutil
import zipimport

print("Python:", sys.version)

def run(cmd):
    print("\n$", " ".join(cmd))
    subprocess.check_call(cmd)

# ------------------------------------------------------------
# 1. Patch Python 3.12 pkg_resources issue BEFORE imports
# ------------------------------------------------------------
# Some Colab/system pkg_resources versions expect pkgutil.ImpImporter,
# which was removed in Python 3.12.
if not hasattr(pkgutil, "ImpImporter"):
    pkgutil.ImpImporter = zipimport.zipimporter

# ------------------------------------------------------------
# 2. Keep setuptools modern enough for Python 3.12,
#    but below torch's <82 constraint
# ------------------------------------------------------------
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "wheel", "setuptools==80.9.0"])

# ------------------------------------------------------------
# 3. Install Basic Pitch dependencies manually
#    This avoids pip backtracking into old basic-pitch/numpy versions.
# ------------------------------------------------------------
run([sys.executable, "-m", "pip", "install", "-q",
     "librosa>=0.10",
     "soundfile",
     "mir-eval",
     "pretty_midi",
     "resampy==0.4.2",
     "onnxruntime"])

# ------------------------------------------------------------
# 4. Force Basic Pitch latest without dependency resolver chaos
# ------------------------------------------------------------
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "basic-pitch==0.4.0"])

# ------------------------------------------------------------
# 5. Import test
# ------------------------------------------------------------
# Re-apply patch right before import in case anything reset it.
if not hasattr(pkgutil, "ImpImporter"):
    pkgutil.ImpImporter = zipimport.zipimporter

import librosa
import soundfile as sf
from basic_pitch.inference import predict as basic_pitch_predict

print("\n✅ Basic Pitch import successful.")
print("✅ librosa:", librosa.__version__)
print("✅ soundfile import successful.")
print("✅ onnxruntime installed.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

$ /usr/bin/python3 -m pip install -q --upgrade pip wheel setuptools==80.9.0

$ /usr/bin/python3 -m pip install -q librosa>=0.10 soundfile mir-eval pretty_midi resampy==0.4.2 onnxruntime

$ /usr/bin/python3 -m pip install -q --no-deps basic-pitch==0.4.0



✅ Basic Pitch import successful.
✅ librosa: 0.11.0
✅ soundfile import successful.
✅ onnxruntime installed.


In [19]:
# -------------------------
# AUDIO EVAL CONFIG
# -------------------------

# Keep the earlier fretboard-only outputs separate from audio-to-tab outputs.
AUDIO_OUTPUT_DIR = CAPSTONE_ROOT / 'outputs' / 'audio_to_tab_basic_pitch_heldout'
AUDIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Candidate locations for actual audio files.
# Your Drive showed audio under both:
#   /content/drive/MyDrive/Capstone/Audio
#   /content/drive/MyDrive/Capstone/GuitarSet/Audio
AUDIO_ROOT_CANDIDATES = [
    CAPSTONE_ROOT / 'GuitarSet' / 'Audio',
    CAPSTONE_ROOT / 'GuitarSet' / 'AudioFiles',
    CAPSTONE_ROOT / 'FullGuitarSetData' / 'AudioFiles',
    CAPSTONE_ROOT / 'FullGuitarSetData' / 'Audio',
    CAPSTONE_ROOT / 'Audio',
    CAPSTONE_ROOT / 'GuitarSet',
    CAPSTONE_ROOT,
    Path('/content/drive/MyDrive/GuitarSet/Audio'),
    Path('/content/drive/MyDrive/GuitarSet'),
]

AUDIO_EXTENSIONS = ['.wav', '.mp3', '.m4a', '.flac', '.ogg']

# Basic Pitch threshold controls how many predicted notes enter the tab algorithm.
# Lower = more notes, higher recall, more false positives.
# Higher = fewer notes, cleaner predictions, lower recall.
BASIC_PITCH_AMPLITUDE_THRESHOLD = 0.40  # raised from 0.30 to reduce noisy/ghost notes
BASIC_PITCH_MIN_MIDI = 40   # low E2
BASIC_PITCH_MAX_MIDI = 88   # high-ish guitar range

# Matching predicted notes to GuitarSet ground truth.
AUDIO_MATCH_ONSET_TOLERANCE_SECONDS = 0.05

# Start small while debugging. Set to None for all matched held-out test recordings.
# Recommended workflow: run with 5 first; after it succeeds, change to None and rerun cells 35 onward.
MAX_AUDIO_RECORDINGS = None

# Honest full audio pipeline settings.
# Keep these False for final reporting.
USE_GROUND_TRUTH_KEY_FOR_CONTEXT = False
USE_GROUND_TRUTH_CHORDS_FOR_CONTEXT = False

# These are audio/predicted-note-derived context signals.
USE_AUDIO_KEY_DETECTION = True
USE_AUDIO_CHORD_DETECTION = True
CHORD_WINDOW_SECONDS = 1.0
CHORD_HOP_SECONDS = 0.5
MIN_NOTES_PER_CHORD_WINDOW = 2

print('AUDIO_OUTPUT_DIR:', AUDIO_OUTPUT_DIR.resolve())
print('\nAudio root candidates:')
for p in AUDIO_ROOT_CANDIDATES:
    print(f' - {p} | exists: {Path(p).exists()}')

print('\nFinal eval split:')
print('USE_HELDOUT_SPLIT:', USE_HELDOUT_SPLIT)
print('TEST_RECORDS:', len(TEST_RECORDS) if 'TEST_RECORDS' in globals() else 'not defined')


AUDIO_OUTPUT_DIR: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/audio_to_tab_basic_pitch_heldout

Audio root candidates:
 - /content/drive/MyDrive/Capstone/GuitarSet/Audio | exists: True
 - /content/drive/MyDrive/Capstone/GuitarSet/AudioFiles | exists: False
 - /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles | exists: True
 - /content/drive/MyDrive/Capstone/FullGuitarSetData/Audio | exists: False
 - /content/drive/MyDrive/Capstone/Audio | exists: True
 - /content/drive/MyDrive/Capstone/GuitarSet | exists: True
 - /content/drive/MyDrive/Capstone | exists: True
 - /content/drive/MyDrive/GuitarSet/Audio | exists: False
 - /content/drive/MyDrive/GuitarSet | exists: False

Final eval split:
USE_HELDOUT_SPLIT: True
TEST_RECORDS: 54



## Pair Held-Out Test Records with Audio Files

This uses the same held-out test set from the tuned notebook. It searches for audio stems like:

```text
00_Jazz3-150-D_comp.jams      ↔ 00_Jazz3-150-D_comp_mic.wav
00_Jazz3-150-D_comp.jams      ↔ 00_Jazz3-150-D_comp_hex.wav
00_Jazz3-150-D_comp.jams      ↔ 00_Jazz3-150-D_comp.wav
```


In [20]:
def find_audio_files(audio_roots, exts=AUDIO_EXTENSIONS):
    """Return dict stem -> path for audio files under candidate roots."""
    audio_by_stem = {}
    for root in audio_roots:
        root = Path(root)
        if not root.exists():
            continue
        for ext in exts:
            try:
                for p in root.rglob(f'*{ext}'):
                    # Prefer first occurrence; exact stem matching is what matters.
                    audio_by_stem.setdefault(p.stem, p)
            except Exception as e:
                print(f'Could not search {root}: {e}')
    return audio_by_stem

AUDIO_BY_STEM = find_audio_files(AUDIO_ROOT_CANDIDATES)
print(f'Found {len(AUDIO_BY_STEM)} audio files.')
for k, v in list(AUDIO_BY_STEM.items())[:15]:
    print(' -', k, '->', v)

def audio_candidates_for_recording(recording):
    """Return likely audio stems for a GuitarSet JAMS stem."""
    recording = str(recording)
    cands = [
        recording,
        f'{recording}_mic',
        f'{recording}_hex',
        recording.replace('_solo', '_solo_mic'),
        recording.replace('_comp', '_comp_mic'),
        recording.replace('_solo', '_solo_hex'),
        recording.replace('_comp', '_comp_hex'),
    ]
    # De-duplicate while preserving order.
    seen, out = set(), []
    for c in cands:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out

def find_audio_for_record(record):
    rec = record['recording']
    for stem in audio_candidates_for_recording(rec):
        if stem in AUDIO_BY_STEM:
            return AUDIO_BY_STEM[stem]
    return None

# Critical: final audio eval uses the held-out TEST_RECORDS, not all records.
AUDIO_EVAL_RECORDS = TEST_RECORDS if USE_HELDOUT_SPLIT else records
AUDIO_EVAL_LABEL = 'heldout_test_audio' if USE_HELDOUT_SPLIT else 'all_records_audio_exploratory'

paired_records = []
missing_audio = []
for record in AUDIO_EVAL_RECORDS:
    audio_path = find_audio_for_record(record)
    if audio_path is None:
        missing_audio.append(record['recording'])
    else:
        paired_records.append((record, audio_path))

print(f'Audio eval set: {AUDIO_EVAL_LABEL}')
print(f'Paired {len(paired_records)} / {len(AUDIO_EVAL_RECORDS)} eval records with audio files.')
for rec, ap in paired_records[:15]:
    print(' -', rec['recording'], '->', ap.name)

if missing_audio:
    print(f'\nMissing audio for {len(missing_audio)} eval records. First few:')
    print(missing_audio[:25])

if not paired_records:
    print('\nNo pairs found. Check AUDIO_ROOT_CANDIDATES and whether audio stems use _mic/_hex suffixes.')


Found 639 audio files.
 - 00_BN1-129-Eb_comp_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_BN1-129-Eb_comp_mic.wav
 - 00_Jazz1-130-D_solo_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_Jazz1-130-D_solo_mic.wav
 - 00_Funk1-97-C_solo_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_Funk1-97-C_solo_mic.wav
 - 00_Funk1-97-C_comp_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_Funk1-97-C_comp_mic.wav
 - 00_Rock1-90-C#_comp_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_Rock1-90-C#_comp_mic.wav
 - 00_Rock1-90-C#_solo_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_Rock1-90-C#_solo_mic.wav
 - 00_Jazz1-130-D_comp_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_Jazz1-130-D_comp_mic.wav
 - 00_SS1-68-E_solo_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_SS1-68-E_solo_mic.wav
 - 00_SS1-68-E_comp_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_SS1-68-E_comp_mic.wav
 - 00_BN1-129-Eb_solo_mic -> /content/dri

In [21]:
# ============================================================
# CAGED-box algorithm, wired into THIS eval harness.
# Reuses existing get_possible_positions / context_cost / group_notes_by_onset /
# enrich_candidate / awkward_fingering_penalty etc. so metrics stay comparable.
# Separate names (..._caged) so combined_all_tuned is left untouched.
# ============================================================
COMFORTABLE_CHORD_SPAN, MAX_CHORD_SPAN = 3, 4   # FIX 1: chord span hard wall at 4

def group_playability_cost_caged(gp):
    if not gp: return 0.0
    strings=[p['string'] for p in gp]; frets=[p['fret'] for p in gp]
    fretted=[f for f in frets if f>0]
    if len(strings)!=len(set(strings)): return float('inf')
    cost=0.0; span=group_span(frets)
    if span>COMFORTABLE_CHORD_SPAN: cost+=2.0*(span-COMFORTABLE_CHORD_SPAN)
    if span>MAX_CHORD_SPAN:        cost+=25.0*(span-MAX_CHORD_SPAN)
    if fretted and min(fretted)<=2 and max(fretted)>=9: cost+=8.0
    if len(strings)>=2:
        ss=max(strings)-min(strings)
        if ss>4 and len(strings)<=3: cost+=1.5*(ss-4)
    hc=estimate_hand_position_from_frets(frets)
    cost+=sum(awkward_fingering_penalty(p,hc) for p in gp)
    if any(f==0 for f in frets) and fretted and max(fretted)>7: cost+=3.0
    return cost

BOX_WINDOW, SHIFT_FREE = 4, 2
BOX_CENTER_COST, BOX_OUTSIDE_COST, OPEN_OUT_OF_BOX_COST = 0.15, 3.00, 0.60
BOX_OFFBOX_COST, BOX_NONHOME_COST, BOX_LOWNECK_COST = 0.60, 1.00, 0.04
CAGED_WEIGHTS={'playability':0.80,'context':0.30,'box_window':1.00,'hand_move':0.70,'position_prior':1.00}
PENTATONIC={'major':[0,2,4,7,9],'minor':[0,3,5,7,10]}
LOW_E_PC=OPEN_STRING_MIDI[0]%12

def parse_key(key_label):
    info=get_key_info(key_label)
    return None if info is None else {'root_pc':int(info['root_pc']),'mode':info['mode'],'scale_pcs':set(info['scale_pcs'])}

def box_anchors_for_key(key,max_fret=MAX_FRET,window=BOX_WINDOW):
    rng=range(0,max_fret-window+1)
    if key is None:
        return [{'anchor':a,'key_cost':BOX_LOWNECK_COST*a} for a in rng]
    r=key['root_pc']; penta=PENTATONIC.get(key['mode'],PENTATONIC['minor'])
    box=set()
    for deg in penta:
        f=(deg+(r-LOW_E_PC))%12
        while f<=max_fret-1: box.add(f); f+=12
    home=set(); h=(r-LOW_E_PC)%12
    while h<=max_fret-1: home.add(h); h+=12
    out=[]
    for a in rng:
        d=min((abs(a-b) for b in box),default=0)
        kc=BOX_OFFBOX_COST*d+(0.0 if a in home else BOX_NONHOME_COST)+BOX_LOWNECK_COST*a
        out.append({'anchor':a,'key_cost':kc})
    return out

def position_window_cost(p,anchor,window=BOX_WINDOW):
    f=p['fret']
    if f==0: return 0.0 if anchor<=2 else OPEN_OUT_OF_BOX_COST
    if anchor<=f<=anchor+window: return BOX_CENTER_COST*abs(f-(anchor+window/2.0))
    return BOX_OUTSIDE_COST*((anchor-f) if f<anchor else (f-(anchor+window)))

def candidate_window_cost(c,anchor): return sum(position_window_cost(p,anchor) for p in c['positions'])

def candidate_groups_caged(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    pls=[]
    for n in group_notes:
        pos=get_possible_positions(n['midi'])
        if not pos: return []
        pls.append(pos)

    def _greedy(cost):   # best-effort: one distinct string per note, lowest fret, keeps note order
        used=set(); pick={}
        for i in sorted(range(len(group_notes)), key=lambda k:-group_notes[k]['midi']):
            opts=sorted(pls[i], key=lambda p:(p['fret'],p['string']))
            chosen=next((p for p in opts if p['string'] not in used), opts[0])
            used.add(chosen['string']); pick[i]=chosen
        return enrich_candidate({'positions':[pick[i] for i in range(len(group_notes))],'base_cost':cost})

    space=1
    for pl in pls: space*=len(pl)
    if space>20000:                       # too many simultaneous-note combos -> skip full product
        return [_greedy(10.0)]

    cands=[]
    for combo in product(*pls):
        combo=list(combo)
        if len(combo)>1 and len({p['string'] for p in combo})!=len(combo): continue
        play=group_playability_cost_caged(combo)
        if not math.isfinite(play): continue
        prior=float(np.mean([position_prior_cost(n['midi'],p) for n,p in zip(group_notes,combo)]))
        base=(CAGED_WEIGHTS['playability']*play
              +CAGED_WEIGHTS['context']*context_cost(group_notes,combo)
              +CAGED_WEIGHTS['position_prior']*prior)
        cands.append(enrich_candidate({'positions':combo,'base_cost':float(base)}))

    if not cands:                         # no conflict-free shape -> keep the record alive
        cands.append(_greedy(100.0))

    return sorted(cands,key=lambda c:c['base_cost'])[:max_candidates]

SOLO_MOVE_SCALE = 0.35   # how much to relax hand-position locking between single notes (solos roam)
SOLO_BOX_SCALE  = 0.30   # how much to relax the home-box pull on single notes (let the prior place them)

def assign_caged_box(notes, weights=CAGED_WEIGHTS, key=None):
    groups = group_notes_by_onset(notes)
    if not groups: return []
    allc = [candidate_groups_caged(g) for g in groups]
    if any(len(c)==0 for c in allc): raise ValueError('group with no candidates')
    is_chord = [len(g) >= 2 for g in groups]                 # chord vs single-note onset

    anchors = box_anchors_for_key(key); A = len(anchors)
    af = np.array([a['anchor'] for a in anchors], dtype=float); n = len(groups)
    ec = np.empty((n, A)); ecand = [[0]*A for _ in range(n)]
    for i, cands in enumerate(allc):
        kc_scale = 1.0 if is_chord[i] else SOLO_BOX_SCALE      # weaken box pull on single notes
        for j, anc in enumerate(anchors):
            a = anc['anchor']; best = None; bci = 0
            for ci, c in enumerate(cands):
                tot = c['base_cost'] + weights['box_window']*candidate_window_cost(c, a)
                if best is None or tot < best: best, bci = tot, ci
            ec[i, j] = best + kc_scale*anc['key_cost']; ecand[i][j] = bci

    delta = np.abs(af[:,None]-af[None,:])
    base_trans = weights['hand_move']*np.maximum(delta-SHIFT_FREE,0.0) + 0.5*np.maximum(delta-LARGE_JUMP_THRESHOLD,0.0)**2
    dp = np.empty((n, A)); back = np.zeros((n, A), dtype=int); dp[0] = ec[0]; back[0] = -1
    for i in range(1, n):
        chordy = is_chord[i] or is_chord[i-1]                 # full lock near chords, loose between single notes
        step_trans = base_trans if chordy else base_trans*SOLO_MOVE_SCALE
        scores = dp[i-1][:,None] + step_trans + ec[i][None,:]
        back[i] = np.argmin(scores, axis=0); dp[i] = scores[back[i], np.arange(A)]

    j = int(np.argmin(dp[-1])); chosen = [j]
    for i in range(n-1, 0, -1): j = int(back[i][j]); chosen.append(j)
    chosen = list(reversed(chosen)); pred = []
    for i, (g, cands) in enumerate(zip(groups, allc)):
        c = cands[ecand[i][chosen[i]]]
        for note, p in zip(g, c['positions']):
            row = dict(note); row.update({'pred_string': p['string'], 'pred_fret': p['fret'],
                                          'method': 'caged_box', 'anchor': anchors[chosen[i]]['anchor']})
            pred.append(row)
    return sorted(pred, key=lambda x: (x['start'], x['midi']))

def drop_octave_harmonics(notes,tol=ONSET_TOLERANCE_SECONDS):
    kept=[]
    for g in group_notes_by_onset(notes):
        midis={n['midi'] for n in g}
        for n in g:
            harm=n['midi']-12 if (n['midi']-12) in midis else (n['midi']-24 if (n['midi']-24) in midis else None)
            if harm is not None:
                amp_n=n.get('amplitude') or 1.0
                amp_low=max((m.get('amplitude') or 1.0) for m in g if m['midi']==harm)
                if amp_n<=amp_low: continue
            kept.append(n)
    return sorted(kept,key=lambda x:(x['start'],x['midi']))

def assign_caged_box_eval(notes):
    # notes=drop_octave_harmonics(notes)   # OFF for fair comparison; keep ON for real inference
    notes=[n for n in notes if get_possible_positions(n['midi'])]
    if not notes: return []
    return assign_caged_box(notes,key=parse_key(notes[0].get('key_label')))

print('CAGED-box eval ready -> add "caged_box": assign_caged_box_eval to AUDIO_ASSIGNMENT_METHODS')

CAGED-box eval ready -> add "caged_box": assign_caged_box_eval to AUDIO_ASSIGNMENT_METHODS


In [22]:
# ============================================================
# Chord-voicing library: reward candidate placements that form a known hand shape.
# Plugs a bonus into the existing caged candidate cost. Only affects chord onsets.
# ============================================================
VOICING_SHAPES = [   # string idx 0=lowE..5=highE; fret offsets relative to lowest fretted note
    {'name':'E-maj', 'offsets':{0:0,1:2,2:2,3:1,4:0,5:0}, 'power':False},
    {'name':'A-maj', 'offsets':{1:0,2:2,3:2,4:2,5:0},     'power':False},
    {'name':'D-maj', 'offsets':{2:0,3:2,4:3,5:2},         'power':False},
    {'name':'C-maj', 'offsets':{1:3,2:2,3:0,4:1,5:0},     'power':False},
    {'name':'G-maj', 'offsets':{0:3,1:2,2:0,3:0,4:0,5:3}, 'power':False},
    {'name':'E-min', 'offsets':{0:0,1:2,2:2,3:0,4:0,5:0}, 'power':False},
    {'name':'A-min', 'offsets':{1:0,2:2,3:2,4:1,5:0},     'power':False},
    {'name':'E-7',   'offsets':{0:0,1:2,2:0,3:1,4:0,5:0}, 'power':False},
    {'name':'A-7',   'offsets':{1:0,2:2,3:0,4:2,5:0},     'power':False},
    {'name':'E-m7',  'offsets':{0:0,1:2,2:0,3:0,4:0,5:0}, 'power':False},
    {'name':'A-m7',  'offsets':{1:0,2:2,3:0,4:1,5:0},     'power':False},
    {'name':'Emaj7', 'offsets':{0:0,1:2,2:1,3:1,4:0,5:0}, 'power':False},
    {'name':'Amaj7', 'offsets':{1:0,2:2,3:1,4:2,5:0},     'power':False},
    {'name':'5-E',   'offsets':{0:0,1:2},       'power':True},
    {'name':'5-A',   'offsets':{1:0,2:2},       'power':True},
    {'name':'5-D',   'offsets':{2:0,3:2},       'power':True},
    {'name':'5-E-oct','offsets':{0:0,1:2,2:2},  'power':True},
    {'name':'5-A-oct','offsets':{1:0,2:2,3:2},  'power':True},
    {'name':'5-D-oct','offsets':{2:0,3:2,4:2},  'power':True},
    {'name':'oct-E', 'offsets':{0:0,2:2},       'power':True},
    {'name':'oct-A', 'offsets':{1:0,3:2},       'power':True},
]

def _off_from_min(d):
    m = min(d.values()); return {k: v - m for k, v in d.items()}

def voicing_bonus(positions):
    """0.0 if the placement isn't a recognized shape; 0.6..1.0 if it is (higher = fuller match).
    Transposition-invariant; matches partial chords; 2-note groups only match power/octave shapes."""
    pts = {p['string']: p['fret'] for p in positions}
    strings = sorted(pts)
    if len(strings) < 2: return 0.0
    cand_off = _off_from_min(pts); n = len(strings); best = 0.0
    for sh in VOICING_SHAPES:
        if (n < 2) if sh['power'] else (n < 3): continue
        smap = sh['offsets']
        if not all(s in smap for s in strings): continue
        if _off_from_min({s: smap[s] for s in strings}) == cand_off:
            best = max(best, 0.6 + 0.4 * (n / len(smap)))
    return best

CAGED_WEIGHTS['voicing'] = 1.0   # tune on validation; try 0.5–2.0

def candidate_groups_voiced(group_notes, max_candidates=MAX_GROUP_CANDIDATES):
    # pull a wider pool so the bonus can promote a shape the base ranking would have truncated
    pool = candidate_groups_caged(group_notes, max_candidates=max(max_candidates * 3, 24))
    w = CAGED_WEIGHTS.get('voicing', 0.0)
    if w and len(group_notes) >= 2:
        for c in pool:
            b = voicing_bonus(c['positions'])
            if b: c['base_cost'] = c['base_cost'] - w * b
        pool = sorted(pool, key=lambda c: c['base_cost'])
    return pool[:max_candidates]

def assign_caged_voiced(notes, weights=CAGED_WEIGHTS, key=None):
    groups = group_notes_by_onset(notes)
    if not groups: return []
    allc = [candidate_groups_voiced(g) for g in groups]          # <-- only change vs caged_box
    if any(len(c) == 0 for c in allc): raise ValueError('group with no candidates')
    is_chord = [len(g) >= 2 for g in groups]
    anchors = box_anchors_for_key(key); A = len(anchors)
    af = np.array([a['anchor'] for a in anchors], dtype=float); n = len(groups)
    ec = np.empty((n, A)); ecand = [[0]*A for _ in range(n)]
    for i, cands in enumerate(allc):
        kc_scale = 1.0 if is_chord[i] else SOLO_BOX_SCALE
        for j, anc in enumerate(anchors):
            a = anc['anchor']; best = None; bci = 0
            for ci, c in enumerate(cands):
                tot = c['base_cost'] + weights['box_window']*candidate_window_cost(c, a)
                if best is None or tot < best: best, bci = tot, ci
            ec[i, j] = best + kc_scale*anc['key_cost']; ecand[i][j] = bci
    delta = np.abs(af[:,None]-af[None,:])
    base_trans = weights['hand_move']*np.maximum(delta-SHIFT_FREE,0.0) + 0.5*np.maximum(delta-LARGE_JUMP_THRESHOLD,0.0)**2
    dp = np.empty((n, A)); back = np.zeros((n, A), dtype=int); dp[0] = ec[0]; back[0] = -1
    for i in range(1, n):
        chordy = is_chord[i] or is_chord[i-1]
        step_trans = base_trans if chordy else base_trans*SOLO_MOVE_SCALE
        scores = dp[i-1][:,None] + step_trans + ec[i][None,:]
        back[i] = np.argmin(scores, axis=0); dp[i] = scores[back[i], np.arange(A)]
    j = int(np.argmin(dp[-1])); chosen = [j]
    for i in range(n-1, 0, -1): j = int(back[i][j]); chosen.append(j)
    chosen = list(reversed(chosen)); pred = []
    for i, (g, cands) in enumerate(zip(groups, allc)):
        c = cands[ecand[i][chosen[i]]]
        for note, p in zip(g, c['positions']):
            row = dict(note); row.update({'pred_string': p['string'], 'pred_fret': p['fret'],
                                          'method': 'caged_voiced', 'anchor': anchors[chosen[i]]['anchor']})
            pred.append(row)
    return sorted(pred, key=lambda x: (x['start'], x['midi']))

def assign_caged_voiced_eval(notes):
    notes = [n for n in notes if get_possible_positions(n['midi'])]
    if not notes: return []
    return assign_caged_voiced(notes, key=parse_key(notes[0].get('key_label')))

print('caged_voiced ready. VOICING_SHAPES:', len(VOICING_SHAPES), '| weight:', CAGED_WEIGHTS['voicing'])

caged_voiced ready. VOICING_SHAPES: 21 | weight: 1.0



## Audio → Basic Pitch Notes + Audio-Derived Key/Chord Context

These functions do **not** use JAMS notes as input. They build the model-input record from audio-derived Basic Pitch notes, then optionally estimate key and chord context from audio/predicted notes.


In [23]:
BASIC_PITCH_CACHE_DIR = AUDIO_OUTPUT_DIR / 'basic_pitch_note_cache'
BASIC_PITCH_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Default context toggles if they were not defined in the config cell.
# By default, this is a full audio-driven pipeline: no GT key/chords as input.
try:
    USE_GROUND_TRUTH_KEY_FOR_CONTEXT
except NameError:
    USE_GROUND_TRUTH_KEY_FOR_CONTEXT = False
try:
    USE_GROUND_TRUTH_CHORDS_FOR_CONTEXT
except NameError:
    USE_GROUND_TRUTH_CHORDS_FOR_CONTEXT = False
try:
    USE_AUDIO_KEY_DETECTION
except NameError:
    USE_AUDIO_KEY_DETECTION = True
try:
    USE_AUDIO_CHORD_DETECTION
except NameError:
    USE_AUDIO_CHORD_DETECTION = True
try:
    CHORD_WINDOW_SECONDS
except NameError:
    CHORD_WINDOW_SECONDS = 1.0
try:
    CHORD_HOP_SECONDS
except NameError:
    CHORD_HOP_SECONDS = 0.5
try:
    MIN_NOTES_PER_CHORD_WINDOW
except NameError:
    MIN_NOTES_PER_CHORD_WINDOW = 2

def midi_to_note_name_simple(midi):
    names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    midi = int(round(midi))
    return f"{names[midi % 12]}{midi // 12 - 1}"

def _basic_pitch_cache_path(audio_path):
    audio_path = Path(audio_path)
    safe_name = audio_path.stem.replace('/', '_')
    return BASIC_PITCH_CACHE_DIR / f'{safe_name}_bp_notes.csv'

def run_basic_pitch_notes(audio_path,
                          amplitude_threshold=BASIC_PITCH_AMPLITUDE_THRESHOLD,
                          min_midi=BASIC_PITCH_MIN_MIDI,
                          max_midi=BASIC_PITCH_MAX_MIDI,
                          use_cache=True):
    """Run Basic Pitch and return note rows in the schema expected by the fretboard algorithm."""
    audio_path = Path(audio_path)
    cache_path = _basic_pitch_cache_path(audio_path)

    if use_cache and cache_path.exists():
        df = pd.read_csv(cache_path)
        return df.to_dict('records')

    print(f'Running Basic Pitch on: {audio_path.name}')
    _, _, note_events = basic_pitch_predict(str(audio_path))

    notes = []
    for event in note_events:
        # Basic Pitch usually returns: start, end, pitch_midi, amplitude, bends
        start, end, pitch_midi, amplitude = event[0], event[1], event[2], event[3]
        if float(amplitude) < amplitude_threshold:
            continue
        midi = int(round(float(pitch_midi)))
        if midi < min_midi or midi > max_midi:
            continue
        notes.append({
            'start': float(start),
            'duration': float(end - start),
            'midi': midi,
            'pitch_class': midi % 12,
            'note_name': midi_to_note_name_simple(midi),
            'amplitude': float(amplitude),
            # Ground truth unknown for audio-derived predictions.
            'true_string': None,
            'true_fret': None,
            'source': 'basic_pitch',
        })

    notes = sorted(notes, key=lambda n: (n['start'], n['midi']))
    pd.DataFrame(notes).to_csv(cache_path, index=False)
    return notes

# Krumhansl-Schmuckler key profiles for simple audio key estimation.
_MAJOR_PROFILE = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
_MINOR_PROFILE = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])

def detect_key_from_audio(audio_path):
    y, sr = librosa.load(str(audio_path), sr=None, mono=True)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    chroma_avg = np.nan_to_num(np.mean(chroma, axis=1))
    scores = []
    for tonic in range(12):
        major_key = f'{PC_TO_NOTE[tonic]} major'
        minor_key = f'{PC_TO_NOTE[tonic]} minor'
        major_score = np.corrcoef(chroma_avg, np.roll(_MAJOR_PROFILE, tonic))[0, 1]
        minor_score = np.corrcoef(chroma_avg, np.roll(_MINOR_PROFILE, tonic))[0, 1]
        scores.append((major_key, major_score))
        scores.append((minor_key, minor_score))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return {'key': scores[0][0], 'score': float(scores[0][1]), 'top5': scores[:5]}

# Lightweight chord templates. These are inferred from Basic Pitch notes, not JAMS chords.
_CHORD_TEMPLATES = []
for root in range(12):
    _CHORD_TEMPLATES.extend([
        {'label': PC_TO_NOTE[root],       'root': root, 'quality': 'maj',  'tones': {(root + x) % 12 for x in [0, 4, 7]}},
        {'label': PC_TO_NOTE[root] + 'm', 'root': root, 'quality': 'min',  'tones': {(root + x) % 12 for x in [0, 3, 7]}},
        {'label': PC_TO_NOTE[root] + '7', 'root': root, 'quality': 'dom7', 'tones': {(root + x) % 12 for x in [0, 4, 7, 10]}},
    ])

def best_chord_for_pitch_classes(pitch_classes):
    """Return the best lightweight chord template for a set/list of pitch classes.

    This keeps dictionary objects OUT of the comparison tuple. Otherwise,
    Python can crash on ties with:
    TypeError: '>' not supported between instances of 'dict' and 'dict'.
    """
    pcs = set(int(pc) % 12 for pc in pitch_classes)
    if len(pcs) < 2:
        return None

    best_score_tuple = None
    best_template = None

    for templ_idx, templ in enumerate(_CHORD_TEMPLATES):
        tones = templ['tones']
        overlap = len(pcs & tones)
        missing = len(tones - pcs)
        extra = len(pcs - tones)
        root_bonus = 0.35 if templ['root'] in pcs else 0.0
        score = overlap - 0.45 * missing - 0.25 * extra + root_bonus

        # Compare only numeric values. The final -templ_idx is a deterministic tie-breaker.
        score_tuple = (score, overlap, -missing, -extra, -templ_idx)

        if best_score_tuple is None or score_tuple > best_score_tuple:
            best_score_tuple = score_tuple
            best_template = templ

    if best_template is None or best_score_tuple[1] < 2:
        return None

    return best_template

def detect_chords_from_basic_pitch_notes(notes, window_seconds=CHORD_WINDOW_SECONDS, hop_seconds=CHORD_HOP_SECONDS):
    """Detect rough chord context from Basic Pitch note pitch classes in sliding windows."""
    if not notes:
        return []
    max_time = max(float(n['start']) + float(n.get('duration', 0.0) or 0.0) for n in notes)
    chords = []
    t = 0.0
    current = None
    prev_label = None

    while t <= max_time:
        t_end = t + window_seconds
        pcs = []
        for n in notes:
            n_start = float(n['start'])
            n_end = n_start + float(n.get('duration', 0.0) or 0.0)
            if n_start < t_end and n_end >= t:
                pcs.append(int(n['pitch_class']))

        templ = best_chord_for_pitch_classes(pcs) if len(pcs) >= MIN_NOTES_PER_CHORD_WINDOW else None
        label = None if templ is None else templ['label']

        if label is not None:
            parsed = {'root': templ['root'], 'quality': templ['quality'], 'tones': sorted(templ['tones'])}
            if current is not None and label == prev_label:
                current['end'] = t_end
                current['duration'] = current['end'] - current['start']
            else:
                if current is not None:
                    chords.append(current)
                current = {
                    'start': float(t),
                    'end': float(t_end),
                    'duration': float(window_seconds),
                    'chord': label,
                    'parsed': parsed,
                    'source': 'basic_pitch_window_chords',
                }
                prev_label = label
        else:
            if current is not None:
                chords.append(current)
                current = None
            prev_label = None

        t += hop_seconds

    if current is not None:
        chords.append(current)
    return chords

def make_audio_record_from_gt(record, audio_path):
    """Build a model-input record from audio-derived notes/context. GT is used only later for scoring."""
    bp_notes = run_basic_pitch_notes(audio_path)

    if USE_GROUND_TRUTH_KEY_FOR_CONTEXT:
        key = record.get('key') or infer_key_from_filename(record['recording'])
        key_source = 'ground_truth_jams_or_filename'
    elif USE_AUDIO_KEY_DETECTION:
        try:
            key_pred = detect_key_from_audio(audio_path)
            key = key_pred['key']
            key_source = 'audio_chroma'
        except Exception as e:
            print(f'Audio key detection failed for {Path(audio_path).name}: {e}')
            key = infer_key_from_filename(record['recording'])
            key_source = 'filename_fallback'
    else:
        key = infer_key_from_filename(record['recording'])
        key_source = 'filename_fallback'

    if USE_GROUND_TRUTH_CHORDS_FOR_CONTEXT:
        chords = record.get('chords', [])
        chord_source = 'ground_truth_jams'
    elif USE_AUDIO_CHORD_DETECTION:
        chords = detect_chords_from_basic_pitch_notes(bp_notes)
        chord_source = 'basic_pitch_window_chords' if chords else 'none_detected'
    else:
        chords = []
        chord_source = 'none'

    return {
        'recording': record['recording'],
        'path': str(audio_path),
        'notes': bp_notes,
        'chords': chords,
        'beats': [],
        'tempo': record.get('tempo'),
        'key': key,
        'key_source': key_source,
        'chord_source': chord_source,
    }



## End-to-End Audio Matching Metrics

A correct full-pipeline tab true positive requires:

```text
correct pitch + onset match + correct string + correct fret
```

The main final metric is `exact_tab_f1`. The `tab_accuracy_given_pitch_match` metric isolates the fretboard assignment quality only on notes Basic Pitch got close enough to match.


In [24]:
def match_audio_predictions_to_truth(pred_rows, gt_notes, onset_tolerance=AUDIO_MATCH_ONSET_TOLERANCE_SECONDS, require_pitch=True):
    """Greedy one-to-one matching by onset and, optionally, MIDI pitch."""
    gt = [
        dict(g, _gt_idx=i)
        for i, g in enumerate(gt_notes)
        if g.get('true_string') is not None
        and g.get('true_fret') is not None
        and 0 <= int(g['true_fret']) <= MAX_FRET
    ]
    pred = [dict(p, _pred_idx=i) for i, p in enumerate(pred_rows)]

    candidates = []
    for pi, p in enumerate(pred):
        if p.get('midi') is None:
            continue
        for gi, g in enumerate(gt):
            if require_pitch and int(p['midi']) != int(g['midi']):
                continue
            dt = abs(float(p['start']) - float(g['start']))
            if dt <= onset_tolerance:
                candidates.append((dt, pi, gi))

    candidates.sort(key=lambda x: x[0])
    used_p, used_g, matches = set(), set(), []

    for dt, pi, gi in candidates:
        if pi in used_p or gi in used_g:
            continue
        used_p.add(pi)
        used_g.add(gi)
        p = pred[pi]
        g = gt[gi]
        pred_string = p.get('pred_string')
        pred_fret = p.get('pred_fret')
        true_string = g.get('true_string')
        true_fret = g.get('true_fret')
        matches.append({
            'recording': p.get('recording'),
            'pred_start': p.get('start'),
            'gt_start': g.get('start'),
            'onset_error': dt,
            'midi': p.get('midi'),
            'pred_string': pred_string,
            'pred_fret': pred_fret,
            'true_string': true_string,
            'true_fret': true_fret,
            'pred_amplitude': p.get('amplitude'),
            'exact_tab_correct': (pred_string == true_string) and (pred_fret == true_fret),
            'string_correct': pred_string == true_string,
            'fret_correct': pred_fret == true_fret,
        })

    return matches, pred, gt

def evaluate_audio_to_tab_record(record, audio_path, assignment_fn, method_name):
    """Run actual audio -> Basic Pitch -> fretboard assignment -> GT matching for one record."""
    t0 = pd.Timestamp.now()
    audio_record = make_audio_record_from_gt(record, audio_path)
    pred_notes_context = enrich_notes_with_context(audio_record)

    if not pred_notes_context:
        assigned_rows = []
    else:
        assigned_rows = assignment_fn(pred_notes_context)

    for r in assigned_rows:
        r['recording'] = record['recording']
        r['method'] = method_name

    matches, pred_all, gt_all = match_audio_predictions_to_truth(
        assigned_rows,
        record['notes'],
        onset_tolerance=AUDIO_MATCH_ONSET_TOLERANCE_SECONDS,
        require_pitch=True,
    )

    n_pred = len(pred_all)
    n_gt = len(gt_all)
    n_pitch_matches = len(matches)

    pitch_precision = n_pitch_matches / n_pred if n_pred else 0.0
    pitch_recall = n_pitch_matches / n_gt if n_gt else 0.0
    pitch_f1 = (2 * pitch_precision * pitch_recall / (pitch_precision + pitch_recall)) if (pitch_precision + pitch_recall) else 0.0

    matches_df = pd.DataFrame(matches)
    exact_tab_tp = int(matches_df['exact_tab_correct'].sum()) if len(matches_df) else 0
    exact_tab_precision = exact_tab_tp / n_pred if n_pred else 0.0
    exact_tab_recall = exact_tab_tp / n_gt if n_gt else 0.0
    exact_tab_f1 = (2 * exact_tab_precision * exact_tab_recall / (exact_tab_precision + exact_tab_recall)) if (exact_tab_precision + exact_tab_recall) else 0.0

    if len(matches_df):
        tab_acc_given_pitch = float(matches_df['exact_tab_correct'].mean())
        string_acc_given_pitch = float(matches_df['string_correct'].mean())
        fret_acc_given_pitch = float(matches_df['fret_correct'].mean())
        mean_onset_error = float(matches_df['onset_error'].mean())
    else:
        tab_acc_given_pitch = 0.0
        string_acc_given_pitch = 0.0
        fret_acc_given_pitch = 0.0
        mean_onset_error = np.nan

    runtime_sec = (pd.Timestamp.now() - t0).total_seconds()

    metrics = {
        'recording': record['recording'],
        'method': method_name,
        'audio_path': str(audio_path),
        'is_solo': record['recording'].endswith('_solo'),
        'is_comp': record['recording'].endswith('_comp'),
        'n_gt_notes': n_gt,
        'n_basic_pitch_notes': n_pred,
        'n_pitch_onset_matches': n_pitch_matches,
        'pitch_precision': pitch_precision,
        'pitch_recall': pitch_recall,
        'pitch_f1': pitch_f1,
        'exact_tab_tp': exact_tab_tp,
        'exact_tab_precision': exact_tab_precision,
        'exact_tab_recall': exact_tab_recall,
        'exact_tab_f1': exact_tab_f1,
        'tab_accuracy_given_pitch_match': tab_acc_given_pitch,
        'string_accuracy_given_pitch_match': string_acc_given_pitch,
        'fret_accuracy_given_pitch_match': fret_acc_given_pitch,
        'mean_onset_error_for_pitch_matches': mean_onset_error,
        'key_source': audio_record.get('key_source'),
        'chord_source': audio_record.get('chord_source'),
        'runtime_sec': runtime_sec,
    }

    assigned_df = pd.DataFrame(assigned_rows)
    if not assigned_df.empty:
        assigned_df['recording'] = record['recording']
        assigned_df['method'] = method_name

    if not matches_df.empty:
        matches_df['recording'] = record['recording']
        matches_df['method'] = method_name

    return metrics, assigned_df, matches_df


## 13b. Strict Conservative Tab Repair Layer

This is the safer version after the first repair test. The previous repair pass slightly hurt GuitarSet accuracy because it smoothed too many playable-but-different positions.

This version only rewrites string/fret choices when the current output is clearly wrong: invalid positions, MIDI/string/fret mismatch, duplicate strings inside a chord, extremely wide chord shapes, or a one-off position outlier between nearby surrounding positions. It preserves MIDI notes, timing, duration, and note count.

Use the repaired methods below to compare against the original methods:

```text
caged_box             vs caged_box_repaired
caged_voiced          vs caged_voiced_repaired
combined_all_tuned    vs combined_all_tuned_repaired
```


In [25]:
# ============================================================
# Conservative constrained tab repair layer for notebook outputs
# ============================================================
# This version is intentionally much stricter than the first repair pass.
# It only rewrites string/fret choices when the current assignment is clearly bad:
#   1) invalid string/fret
#   2) string/fret does not reproduce the MIDI pitch
#   3) simultaneous notes are assigned to the same string
#   4) extremely wide chord shape
#   5) isolated position outlier between nearby surrounding positions
#
# It does NOT broadly smooth normal fret jumps. That earlier behavior slightly hurt
# GuitarSet accuracy, so this version avoids changing playable-but-different choices.

from itertools import product

REPAIR_ENABLED = True
REPAIR_MODE = 'strict_conservative'

# Only flag truly ugly chord spans. Normal guitar voicings can be wider than 5 frets.
REPAIR_SEVERE_CHORD_SPAN = 9
REPAIR_TARGET_MAX_CHORD_SPAN = 7

# Isolated outlier logic: current position must be far from both previous and next,
# while previous and next are near each other. This catches obvious one-off glitches
# without penalizing real position shifts.
REPAIR_ENABLE_ISOLATED_OUTLIER_FIX = True
REPAIR_OUTLIER_JUMP = 12
REPAIR_NEIGHBOR_AGREEMENT = 3
REPAIR_MIN_OUTLIER_IMPROVEMENT = 5

# Candidate scoring: preserve original unless fixing a hard violation.
REPAIR_ORIGINAL_STRING_CHANGE_COST = 8.0
REPAIR_ORIGINAL_FRET_CHANGE_COST = 2.0
REPAIR_PRIOR_WEIGHT = 0.4
REPAIR_CONTEXT_WEIGHT = 0.1
REPAIR_MAX_CHANGES_TO_PRINT = 30


def _repair_to_int(value):
    try:
        if value is None:
            return None
        if pd.isna(value):
            return None
        return int(round(float(value)))
    except Exception:
        return None


def _repair_pitch_for_position(string_idx, fret):
    s = _repair_to_int(string_idx)
    f = _repair_to_int(fret)
    if s is None or f is None:
        return None
    if not (0 <= s < len(OPEN_STRING_MIDI)):
        return None
    if f < 0 or f > MAX_FRET:
        return None
    return int(OPEN_STRING_MIDI[s]) + int(f)


def _repair_current_position(note):
    s = _repair_to_int(note.get('pred_string'))
    f = _repair_to_int(note.get('pred_fret'))
    if s is None or f is None:
        return None
    if not (0 <= s < len(OPEN_STRING_MIDI)):
        return None
    if not (0 <= f <= MAX_FRET):
        return None
    return {'string': int(s), 'fret': int(f)}


def _repair_note_hard_reasons(note):
    midi = _repair_to_int(note.get('midi'))
    cur = _repair_current_position(note)
    if midi is None:
        return []
    if cur is None:
        return ['invalid_position']
    actual_midi = _repair_pitch_for_position(cur['string'], cur['fret'])
    if actual_midi != midi:
        return ['midi_position_mismatch']
    return []


def _repair_anchor(positions):
    fretted = [p['fret'] for p in positions if p and p.get('fret', 0) > 0]
    if fretted:
        return float(np.median(fretted))
    frets = [p['fret'] for p in positions if p is not None]
    return float(np.median(frets)) if frets else None


def _repair_span(positions):
    fretted = [p['fret'] for p in positions if p and p.get('fret', 0) > 0]
    if len(fretted) <= 1:
        return 0
    return int(max(fretted) - min(fretted))


def _repair_candidate_combos(group):
    position_lists = []
    for note in group:
        midi = _repair_to_int(note.get('midi'))
        if midi is None:
            return []
        positions = get_possible_positions(midi)
        if not positions:
            return []
        position_lists.append(positions)

    combos = []
    for combo in product(*position_lists):
        combo = [dict(p) for p in combo]
        strings = [p['string'] for p in combo]
        # Simultaneous notes cannot share one guitar string.
        if len(strings) != len(set(strings)):
            continue
        combos.append(combo)
    return combos


def _repair_group_reasons_from_positions(group, positions, prev_anchor=None, next_anchor=None):
    reasons = []

    # Hard per-note validation.
    for note, pos in zip(group, positions):
        midi = _repair_to_int(note.get('midi'))
        if midi is None:
            continue
        if pos is None:
            reasons.append('invalid_position')
            continue
        if _repair_pitch_for_position(pos['string'], pos['fret']) != midi:
            reasons.append('midi_position_mismatch')

    if all(p is not None for p in positions):
        strings = [p['string'] for p in positions]
        if len(strings) != len(set(strings)):
            reasons.append('duplicate_string_in_chord')

        span = _repair_span(positions)
        if span > REPAIR_SEVERE_CHORD_SPAN:
            reasons.append('severe_wide_chord_span')

        # Conservative isolated outlier. Requires both neighbors and neighbor agreement.
        if REPAIR_ENABLE_ISOLATED_OUTLIER_FIX and prev_anchor is not None and next_anchor is not None:
            if abs(prev_anchor - next_anchor) <= REPAIR_NEIGHBOR_AGREEMENT:
                anchor = _repair_anchor(positions)
                if anchor is not None:
                    neighbor_anchor = 0.5 * (prev_anchor + next_anchor)
                    if abs(anchor - prev_anchor) >= REPAIR_OUTLIER_JUMP and abs(anchor - next_anchor) >= REPAIR_OUTLIER_JUMP:
                        # Do not treat open-position notes as outliers; those are often intentional.
                        frets = [p['fret'] for p in positions]
                        if not any(f == 0 for f in frets):
                            reasons.append('isolated_position_outlier')

    return sorted(set(reasons))


def _repair_current_positions(group):
    return [_repair_current_position(n) for n in group]


def _repair_group_reasons(group, prev_anchor=None, next_anchor=None):
    return _repair_group_reasons_from_positions(group, _repair_current_positions(group), prev_anchor, next_anchor)


def _repair_candidate_score(combo, group, prev_anchor=None, next_anchor=None, current_anchor=None):
    score = 0.0
    span = _repair_span(combo)
    if span > REPAIR_TARGET_MAX_CHORD_SPAN:
        score += 12.0 * (span - REPAIR_TARGET_MAX_CHORD_SPAN)

    # Preserve original heavily.
    for pos, note in zip(combo, group):
        cur = _repair_current_position(note)
        if cur is not None:
            if pos['string'] != cur['string']:
                score += REPAIR_ORIGINAL_STRING_CHANGE_COST
            score += REPAIR_ORIGINAL_FRET_CHANGE_COST * abs(pos['fret'] - cur['fret'])

        try:
            score += REPAIR_PRIOR_WEIGHT * position_prior_cost(note.get('midi'), pos)
        except Exception:
            pass
        try:
            score += REPAIR_CONTEXT_WEIGHT * context_cost(note, pos)
        except Exception:
            pass

    # For isolated outlier fixes, move toward the agreed surrounding position.
    if prev_anchor is not None and next_anchor is not None and abs(prev_anchor - next_anchor) <= REPAIR_NEIGHBOR_AGREEMENT:
        target = 0.5 * (prev_anchor + next_anchor)
        anchor = _repair_anchor(combo)
        if anchor is not None:
            score += 1.5 * abs(anchor - target)

    return float(score)


def _repair_is_meaningful_improvement(current_reasons, new_reasons, current_positions, new_positions, prev_anchor=None, next_anchor=None):
    current_reasons = set(current_reasons)
    new_reasons = set(new_reasons)

    if not current_reasons:
        return False

    # Never accept a repair that still has the same hard invalidity/mismatch/duplicate issue.
    hard = {'invalid_position', 'midi_position_mismatch', 'duplicate_string_in_chord'}
    if current_reasons & hard:
        return len(new_reasons & hard) == 0

    # Wide chord repair must materially reduce span.
    if 'severe_wide_chord_span' in current_reasons:
        old_span = _repair_span(current_positions)
        new_span = _repair_span(new_positions)
        return new_span <= REPAIR_TARGET_MAX_CHORD_SPAN and new_span < old_span

    # Outlier repair must move toward neighbors by a meaningful amount.
    if 'isolated_position_outlier' in current_reasons and prev_anchor is not None and next_anchor is not None:
        target = 0.5 * (prev_anchor + next_anchor)
        old_anchor = _repair_anchor(current_positions)
        new_anchor = _repair_anchor(new_positions)
        if old_anchor is None or new_anchor is None:
            return False
        return (abs(old_anchor - target) - abs(new_anchor - target)) >= REPAIR_MIN_OUTLIER_IMPROVEMENT

    return False


def repair_notebook_tab_assignments(assigned_rows, method_suffix='_repaired', verbose=False):
    """Repair only clearly bad pred_string/pred_fret choices in notebook assignment rows."""
    if not assigned_rows:
        return [], {
            'enabled': bool(REPAIR_ENABLED),
            'mode': REPAIR_MODE,
            'changed': False,
            'num_notes_changed': 0,
            'num_groups_changed': 0,
            'reasons': ['no_notes_to_repair'],
            'changes': [],
        }

    if not REPAIR_ENABLED:
        return [dict(r) for r in assigned_rows], {
            'enabled': False,
            'mode': REPAIR_MODE,
            'changed': False,
            'num_notes_changed': 0,
            'num_groups_changed': 0,
            'reasons': ['repair_disabled'],
            'changes': [],
        }

    original_rows = [dict(r) for r in assigned_rows]
    groups = group_notes_by_onset(original_rows)

    original_anchors = []
    for g in groups:
        positions = _repair_current_positions(g)
        original_anchors.append(_repair_anchor(positions) if all(p is not None for p in positions) else None)

    repaired_groups = []
    changes = []
    reasons_seen = set()

    for gi, group in enumerate(groups):
        prev_anchor = None
        for earlier in reversed(original_anchors[max(0, gi - 3):gi]):
            if earlier is not None:
                prev_anchor = earlier
                break

        next_anchor = None
        for later in original_anchors[gi + 1: gi + 4]:
            if later is not None:
                next_anchor = later
                break

        current_positions = _repair_current_positions(group)
        reasons = _repair_group_reasons(group, prev_anchor=prev_anchor, next_anchor=next_anchor)
        reasons_seen.update(reasons)

        repaired = [dict(n) for n in group]
        if reasons:
            combos = _repair_candidate_combos(group)
            if combos:
                # Only consider candidates that reduce the detected hard issue.
                scored = []
                for combo in combos:
                    new_reasons = _repair_group_reasons_from_positions(group, combo, prev_anchor=prev_anchor, next_anchor=next_anchor)
                    if _repair_is_meaningful_improvement(reasons, new_reasons, current_positions, combo, prev_anchor, next_anchor):
                        scored.append((_repair_candidate_score(combo, group, prev_anchor, next_anchor), combo, new_reasons))

                if scored:
                    best_score, best_combo, best_new_reasons = min(scored, key=lambda x: x[0])
                    repaired = []
                    for note, pos in zip(group, best_combo):
                        row = dict(note)
                        row['pred_string'] = int(pos['string'])
                        row['pred_fret'] = int(pos['fret'])
                        repaired.append(row)
                else:
                    reasons_seen.add('no_strictly_better_repair_candidate')
            else:
                reasons_seen.add('no_valid_repair_candidate')

        old_positions = [(n.get('pred_string'), n.get('pred_fret')) for n in group]
        new_positions = [(n.get('pred_string'), n.get('pred_fret')) for n in repaired]
        if old_positions != new_positions:
            changes.append({
                'group_start': round(float(group[0].get('start', 0.0)), 3),
                'midi': [int(n.get('midi')) for n in group if n.get('midi') is not None],
                'reasons': reasons,
                'old': old_positions,
                'new': new_positions,
            })
            for row in repaired:
                row['repair_changed_group'] = True
                row['repair_reasons'] = ','.join(reasons)
        else:
            for row in repaired:
                row['repair_changed_group'] = False
                row['repair_reasons'] = ''

        repaired_groups.append(repaired)

    repaired_rows = [r for g in repaired_groups for r in g]
    repaired_rows = sorted(repaired_rows, key=lambda x: (float(x.get('start', 0.0)), int(x.get('midi', 0))))
    original_sorted = sorted(original_rows, key=lambda x: (float(x.get('start', 0.0)), int(x.get('midi', 0))))

    changed_note_count = 0
    changed_lookup = set()
    old_position_lookup = {}
    new_position_lookup = {}
    for old, new in zip(original_sorted, repaired_rows):
        key = (round(float(new.get('start', 0.0)), 6), int(new.get('midi', -999)))
        old_position_lookup[key] = (old.get('pred_string'), old.get('pred_fret'))
        new_position_lookup[key] = (new.get('pred_string'), new.get('pred_fret'))
        changed = (old.get('pred_string') != new.get('pred_string')) or (old.get('pred_fret') != new.get('pred_fret'))
        if changed:
            changed_note_count += 1
            changed_lookup.add(key)

    for row in repaired_rows:
        key = (round(float(row.get('start', 0.0)), 6), int(row.get('midi', -999)))
        old_pos = old_position_lookup.get(key, (None, None))
        new_pos = new_position_lookup.get(key, (row.get('pred_string'), row.get('pred_fret')))
        row['repair_changed_note'] = key in changed_lookup
        row['repair_old_string'] = old_pos[0]
        row['repair_old_fret'] = old_pos[1]
        row['repair_new_string'] = new_pos[0]
        row['repair_new_fret'] = new_pos[1]
        row['repair_num_notes_changed'] = changed_note_count
        row['repair_num_groups_changed'] = len(changes)
        row['repair_reasons_all'] = ','.join(sorted(reasons_seen))
        row['repair_mode'] = REPAIR_MODE

    report = {
        'enabled': True,
        'mode': REPAIR_MODE,
        'changed': bool(changes),
        'num_notes_changed': int(changed_note_count),
        'num_groups_changed': int(len(changes)),
        'reasons': sorted(reasons_seen),
        'changes': changes[:REPAIR_MAX_CHANGES_TO_PRINT],
    }

    if verbose:
        print('Repair report:', report)

    return repaired_rows, report


def make_repaired_assignment_fn(base_assignment_fn, base_method_name):
    """Wrap any existing assignment function with conservative constrained repair."""
    repaired_method_name = f'{base_method_name}_repaired'

    def _wrapped(notes):
        raw_rows = base_assignment_fn(notes)
        repaired_rows, report = repair_notebook_tab_assignments(raw_rows)
        for r in repaired_rows:
            r['method'] = repaired_method_name
            r['repair_enabled'] = report['enabled']
            r['repair_changed_any'] = report['changed']
            r['repair_report_reasons'] = ','.join(report['reasons'])
            r['repair_mode'] = report['mode']
        return repaired_rows

    return _wrapped


print('Loaded STRICT conservative tab repair layer.')
print('Repair mode:', REPAIR_MODE)
print('Notebook string convention: pred_string 0=low E ... 5=high E')


Loaded STRICT conservative tab repair layer.
Repair mode: strict_conservative
Notebook string convention: pred_string 0=low E ... 5=high E



## Run Held-Out Full Audio-to-Tab Evaluation

This is the main end-to-end test. Keep `MAX_AUDIO_RECORDINGS = 5` while debugging. Once it works, set `MAX_AUDIO_RECORDINGS = None` in the audio config cell and rerun from the audio matching cell onward.


In [26]:
AUDIO_ASSIGNMENT_METHODS = {
    # Original methods
    'caged_box':                    assign_caged_box_eval,
    'caged_voiced':                 assign_caged_voiced_eval,
    'combined_all_tuned':           assign_combined_all_tuned,

    # Repaired variants: same detected notes/timing, rewritten string/fret choices only when suspicious
    'caged_box_repaired':           make_repaired_assignment_fn(assign_caged_box_eval, 'caged_box'),
    'caged_voiced_repaired':        make_repaired_assignment_fn(assign_caged_voiced_eval, 'caged_voiced'),
    'combined_all_tuned_repaired':  make_repaired_assignment_fn(assign_combined_all_tuned, 'combined_all_tuned'),
}

records_to_run = paired_records if MAX_AUDIO_RECORDINGS is None else paired_records[:MAX_AUDIO_RECORDINGS]
print(f'Running held-out audio-to-tab evaluation on {len(records_to_run)} recordings from {AUDIO_EVAL_LABEL}.')
print(f'Basic Pitch amplitude threshold: {BASIC_PITCH_AMPLITUDE_THRESHOLD}')
print(f'Onset match tolerance: {AUDIO_MATCH_ONSET_TOLERANCE_SECONDS} sec')
print(f'GT key context: {USE_GROUND_TRUTH_KEY_FOR_CONTEXT}; GT chord context: {USE_GROUND_TRUTH_CHORDS_FOR_CONTEXT}')
print(f'Audio key detection: {USE_AUDIO_KEY_DETECTION}; Audio chord detection: {USE_AUDIO_CHORD_DETECTION}')

all_audio_metrics, all_audio_predictions, all_audio_matches, audio_failed = [], [], [], []

for record, audio_path in records_to_run:
    print(f"\nRecording: {record['recording']} | audio: {audio_path.name}")
    for method_name, assign_fn in AUDIO_ASSIGNMENT_METHODS.items():
        print(f'  - {method_name}')
        try:
            metrics, assigned_df, matches_df = evaluate_audio_to_tab_record(record, audio_path, assign_fn, method_name)
            metrics['eval_set'] = AUDIO_EVAL_LABEL
            all_audio_metrics.append(metrics)
            if not assigned_df.empty:
                all_audio_predictions.append(assigned_df)
            if not matches_df.empty:
                all_audio_matches.append(matches_df)
        except Exception as e:
            print('    FAILED:', repr(e))
            audio_failed.append({
                'recording': record['recording'],
                'method': method_name,
                'audio_path': str(audio_path),
                'error': repr(e),
            })

# Build DataFrames
audio_metrics_df = pd.DataFrame(all_audio_metrics)
audio_predictions_df = pd.concat(all_audio_predictions, ignore_index=True) if all_audio_predictions else pd.DataFrame()
audio_matches_df = pd.concat(all_audio_matches, ignore_index=True) if all_audio_matches else pd.DataFrame()
audio_failed_df = pd.DataFrame(audio_failed)

# Summary weighted by note counts where appropriate.
def _safe_weighted_avg(df, value_col, weight_col):
    if df.empty or value_col not in df or weight_col not in df:
        return np.nan
    weights = df[weight_col].fillna(0).astype(float)
    vals = df[value_col].fillna(0).astype(float)
    return float(np.average(vals, weights=weights)) if weights.sum() else float(vals.mean())

summary_rows = []
if not audio_metrics_df.empty:
    for method, g in audio_metrics_df.groupby('method'):
        n_pred = g['n_basic_pitch_notes'].sum()
        n_gt = g['n_gt_notes'].sum()
        n_pitch = g['n_pitch_onset_matches'].sum()
        exact_tp = g['exact_tab_tp'].sum()

        pitch_precision = n_pitch / n_pred if n_pred else 0.0
        pitch_recall = n_pitch / n_gt if n_gt else 0.0
        pitch_f1 = (2 * pitch_precision * pitch_recall / (pitch_precision + pitch_recall)) if (pitch_precision + pitch_recall) else 0.0

        exact_tab_precision = exact_tp / n_pred if n_pred else 0.0
        exact_tab_recall = exact_tp / n_gt if n_gt else 0.0
        exact_tab_f1 = (2 * exact_tab_precision * exact_tab_recall / (exact_tab_precision + exact_tab_recall)) if (exact_tab_precision + exact_tab_recall) else 0.0

        summary_rows.append({
            'method': method,
            'recordings': g['recording'].nunique(),
            'n_gt_notes': int(n_gt),
            'n_basic_pitch_notes': int(n_pred),
            'n_pitch_onset_matches': int(n_pitch),
            'pitch_precision': pitch_precision,
            'pitch_recall': pitch_recall,
            'pitch_f1': pitch_f1,
            'exact_tab_precision': exact_tab_precision,
            'exact_tab_recall': exact_tab_recall,
            'exact_tab_f1': exact_tab_f1,
            'tab_accuracy_given_pitch_match': _safe_weighted_avg(g, 'tab_accuracy_given_pitch_match', 'n_pitch_onset_matches'),
            'string_accuracy_given_pitch_match': _safe_weighted_avg(g, 'string_accuracy_given_pitch_match', 'n_pitch_onset_matches'),
            'fret_accuracy_given_pitch_match': _safe_weighted_avg(g, 'fret_accuracy_given_pitch_match', 'n_pitch_onset_matches'),
            'mean_runtime_sec_per_recording': float(g['runtime_sec'].mean()),
        })

audio_summary_df = pd.DataFrame(summary_rows).sort_values(['exact_tab_f1', 'pitch_f1'], ascending=[False, False]) if summary_rows else pd.DataFrame()

# Save outputs
audio_metrics_path = AUDIO_OUTPUT_DIR / 'audio_to_tab_eval_by_recording.csv'
audio_summary_path = AUDIO_OUTPUT_DIR / 'audio_to_tab_eval_summary.csv'
audio_predictions_path = AUDIO_OUTPUT_DIR / 'audio_to_tab_predictions_all.csv'
audio_matches_path = AUDIO_OUTPUT_DIR / 'audio_to_tab_matches.csv'
audio_failed_path = AUDIO_OUTPUT_DIR / 'audio_to_tab_failed_runs.csv'

audio_metrics_df.to_csv(audio_metrics_path, index=False)
audio_summary_df.to_csv(audio_summary_path, index=False)
audio_predictions_df.to_csv(audio_predictions_path, index=False)
audio_matches_df.to_csv(audio_matches_path, index=False)
audio_failed_df.to_csv(audio_failed_path, index=False)

print('\nSaved audio outputs:')
for p in [audio_metrics_path, audio_summary_path, audio_predictions_path, audio_matches_path, audio_failed_path]:
    print(' -', p.resolve())

print('\nAudio-to-tab summary:')
if not audio_summary_df.empty:
    display(audio_summary_df.round(4))
else:
    print('No successful audio evaluations yet.')

if not audio_failed_df.empty:
    print('\nFailed audio runs:')
    display(audio_failed_df)


Running held-out audio-to-tab evaluation on 54 recordings from heldout_test_audio.
Basic Pitch amplitude threshold: 0.4
Onset match tolerance: 0.05 sec
GT key context: False; GT chord context: False
Audio key detection: True; Audio chord detection: True

Recording: 02_BN2-131-B_solo | audio: 02_BN2-131-B_solo_mic.wav
  - caged_box
  - caged_voiced
  - combined_all_tuned
  - caged_box_repaired
  - caged_voiced_repaired
  - combined_all_tuned_repaired

Recording: 00_Rock2-142-D_comp | audio: 00_Rock2-142-D_comp_mic.wav
  - caged_box
  - caged_voiced
  - combined_all_tuned
  - caged_box_repaired
  - caged_voiced_repaired
  - combined_all_tuned_repaired

Recording: 05_Rock2-85-F_solo | audio: 05_Rock2-85-F_solo_mic.wav
  - caged_box
  - caged_voiced
  - combined_all_tuned
  - caged_box_repaired
  - caged_voiced_repaired
  - combined_all_tuned_repaired

Recording: 04_Jazz3-137-Eb_comp | audio: 04_Jazz3-137-Eb_comp_mic.wav
  - caged_box
  - caged_voiced
  - combined_all_tuned
  - caged_box_r

,method,recordings,n_gt_notes,n_basic_pitch_notes,n_pitch_onset_matches,pitch_precision,pitch_recall,pitch_f1,exact_tab_precision,exact_tab_recall,exact_tab_f1,tab_accuracy_given_pitch_match,string_accuracy_given_pitch_match,fret_accuracy_given_pitch_match,mean_runtime_sec_per_recording
2,caged_voiced,54,10207,9561,7668,0.802,0.7512,0.7758,0.5549,0.5197,0.5367,0.6918,0.6918,0.6918,0.7144
3,caged_voiced_repaired,54,10207,9561,7668,0.802,0.7512,0.7758,0.5549,0.5197,0.5367,0.6918,0.6918,0.6918,0.7032
0,caged_box,54,10207,9561,7668,0.802,0.7512,0.7758,0.5497,0.5149,0.5318,0.6854,0.6854,0.6854,1.8474
1,caged_box_repaired,54,10207,9561,7668,0.802,0.7512,0.7758,0.5497,0.5149,0.5318,0.6854,0.6854,0.6854,0.6744
4,combined_all_tuned,54,10207,9561,7668,0.802,0.7512,0.7758,0.5191,0.4862,0.5021,0.6472,0.6472,0.6472,0.7060
5,combined_all_tuned_repaired,54,10207,9561,7668,0.802,0.7512,0.7758,0.5191,0.4862,0.5021,0.6472,0.6472,0.6472,0.7205



## 14b. Repair Impact Summary

Run this after the audio evaluation cell. It compares each original method against its repaired version and prints whether the repair improved exact tab F1 and tab accuracy given matched pitches.


In [27]:

# ============================================================
# Repair impact summary: original vs repaired methods
# ============================================================
if 'audio_summary_df' not in globals() or audio_summary_df.empty:
    print('Run the audio evaluation cell first.')
else:
    repair_pairs = []
    methods = set(audio_summary_df['method'].astype(str))
    for m in sorted(methods):
        if m.endswith('_repaired'):
            base = m.replace('_repaired', '')
            if base in methods:
                repair_pairs.append((base, m))

    rows = []
    for base, repaired in repair_pairs:
        b = audio_summary_df[audio_summary_df['method'] == base].iloc[0]
        r = audio_summary_df[audio_summary_df['method'] == repaired].iloc[0]
        rows.append({
            'base_method': base,
            'repaired_method': repaired,
            'base_exact_tab_f1': b['exact_tab_f1'],
            'repaired_exact_tab_f1': r['exact_tab_f1'],
            'delta_exact_tab_f1': r['exact_tab_f1'] - b['exact_tab_f1'],
            'base_tab_acc_given_pitch': b['tab_accuracy_given_pitch_match'],
            'repaired_tab_acc_given_pitch': r['tab_accuracy_given_pitch_match'],
            'delta_tab_acc_given_pitch': r['tab_accuracy_given_pitch_match'] - b['tab_accuracy_given_pitch_match'],
            'base_string_acc': b['string_accuracy_given_pitch_match'],
            'repaired_string_acc': r['string_accuracy_given_pitch_match'],
            'delta_string_acc': r['string_accuracy_given_pitch_match'] - b['string_accuracy_given_pitch_match'],
            'base_fret_acc': b['fret_accuracy_given_pitch_match'],
            'repaired_fret_acc': r['fret_accuracy_given_pitch_match'],
            'delta_fret_acc': r['fret_accuracy_given_pitch_match'] - b['fret_accuracy_given_pitch_match'],
        })

    repair_impact_df = pd.DataFrame(rows)
    if repair_impact_df.empty:
        print('No original/repaired method pairs found in audio_summary_df.')
    else:
        display(repair_impact_df.round(4))
        repair_impact_path = AUDIO_OUTPUT_DIR / 'repair_impact_summary.csv'
        repair_impact_df.to_csv(repair_impact_path, index=False)
        print('Saved repair impact summary to:', repair_impact_path.resolve())

    if 'audio_predictions_df' in globals() and not audio_predictions_df.empty and 'repair_changed_note' in audio_predictions_df.columns:
        changed = audio_predictions_df[audio_predictions_df['repair_changed_note'].fillna(False) == True].copy()
        print(f'Changed-note rows found: {len(changed)}')
        cols = [c for c in [
            'recording', 'method', 'start', 'midi', 'pred_string', 'pred_fret',
            'repair_reasons', 'repair_report_reasons', 'amplitude'
        ] if c in changed.columns]
        if len(changed):
            display(changed[cols].head(50))
            changed_path = AUDIO_OUTPUT_DIR / 'repair_changed_notes.csv'
            changed.to_csv(changed_path, index=False)
            print('Saved changed-note details to:', changed_path.resolve())


,base_method,repaired_method,base_exact_tab_f1,repaired_exact_tab_f1,delta_exact_tab_f1,base_tab_acc_given_pitch,repaired_tab_acc_given_pitch,delta_tab_acc_given_pitch,base_string_acc,repaired_string_acc,delta_string_acc,base_fret_acc,repaired_fret_acc,delta_fret_acc
0,caged_box,caged_box_repaired,0.5318,0.5318,0.0,0.6854,0.6854,0.0,0.6854,0.6854,0.0,0.6854,0.6854,0.0
1,caged_voiced,caged_voiced_repaired,0.5367,0.5367,0.0,0.6918,0.6918,0.0,0.6918,0.6918,0.0,0.6918,0.6918,0.0
2,combined_all_tuned,combined_all_tuned_repaired,0.5021,0.5021,0.0,0.6472,0.6472,0.0,0.6472,0.6472,0.0,0.6472,0.6472,0.0


Saved repair impact summary to: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/audio_to_tab_basic_pitch_heldout/repair_impact_summary.csv
Changed-note rows found: 4


,recording,method,start,midi,pred_string,pred_fret,repair_reasons,repair_report_reasons,amplitude
17483,00_Funk1-97-C_solo,combined_all_tuned_repaired,10.803698,57,2,7,isolated_position_outlier,isolated_position_outlier,0.787104
56287,00_Funk2-108-Eb_solo,caged_box_repaired,17.866416,58,2,8,isolated_position_outlier,isolated_position_outlier,0.839768
56412,00_Funk2-108-Eb_solo,caged_voiced_repaired,17.866416,58,2,8,isolated_position_outlier,isolated_position_outlier,0.839768
56537,00_Funk2-108-Eb_solo,combined_all_tuned_repaired,17.866416,58,2,8,isolated_position_outlier,isolated_position_outlier,0.839768


Saved changed-note details to: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/audio_to_tab_basic_pitch_heldout/repair_changed_notes.csv


In [28]:
# Solo vs comp breakdown, reusing audio_metrics_df / audio_matches_df from the last test run.
def _seg(rec):
    return 'solo' if str(rec).endswith('_solo') else ('comp' if str(rec).endswith('_comp') else 'other')

def segment_breakdown(df):
    d = df.copy()
    d['segment'] = d['recording'].map(_seg)
    rows = []
    for (method, seg), g in d.groupby(['method', 'segment']):
        n_pred = g['n_basic_pitch_notes'].sum(); n_gt = g['n_gt_notes'].sum()
        n_pitch = g['n_pitch_onset_matches'].sum(); tp = g['exact_tab_tp'].sum()
        etp_p = tp / n_pred if n_pred else 0; etp_r = tp / n_gt if n_gt else 0
        rows.append({
            'method': method, 'segment': seg, 'recordings': g['recording'].nunique(),
            'n_gt_notes': int(n_gt), 'n_pitch_matches': int(n_pitch),
            'pitch_recall': n_pitch / n_gt if n_gt else 0,
            'exact_tab_f1': (2*etp_p*etp_r/(etp_p+etp_r)) if (etp_p+etp_r) else 0,
            'tab_accuracy_given_pitch_match': tp / n_pitch if n_pitch else 0,
        })
    return pd.DataFrame(rows).sort_values(['segment', 'method'])

print("=== Position accuracy by solo vs comp ===")
display(segment_breakdown(audio_metrics_df).round(4))

# How wrong are the wrong ones? adjacent-string error vs wild error.
m = audio_matches_df.copy()
m['segment'] = m['recording'].map(_seg)
m['string_err'] = (m['pred_string'] - m['true_string']).abs()
wrong = m[~m['string_correct']]
print("\n=== Among WRONG-string notes: distribution of |pred_string - true_string| ===")
display(wrong.groupby(['method', 'segment'])['string_err']
            .value_counts(normalize=True).unstack(fill_value=0).round(3))

=== Position accuracy by solo vs comp ===


,method,segment,recordings,n_gt_notes,n_pitch_matches,pitch_recall,exact_tab_f1,tab_accuracy_given_pitch_match
0,caged_box,comp,27,7364,5252,0.7132,0.5454,0.7199
2,caged_box_repaired,comp,27,7364,5252,0.7132,0.5454,0.7199
4,caged_voiced,comp,27,7364,5252,0.7132,0.5536,0.7308
6,caged_voiced_repaired,comp,27,7364,5252,0.7132,0.5536,0.7308
8,combined_all_tuned,comp,27,7364,5252,0.7132,0.5059,0.6677
10,combined_all_tuned_repaired,comp,27,7364,5252,0.7132,0.5059,0.6677
1,caged_box,solo,27,2843,2416,0.8498,0.4997,0.6105
3,caged_box_repaired,solo,27,2843,2416,0.8498,0.4997,0.6105
5,caged_voiced,solo,27,2843,2416,0.8498,0.4970,0.6072
7,caged_voiced_repaired,solo,27,2843,2416,0.8498,0.4970,0.6072



=== Among WRONG-string notes: distribution of |pred_string - true_string| ===


string_err                               1      2      3      4      5
method                      segment                                   
caged_box                   comp     0.978  0.020  0.002  0.000  0.001
                            solo     0.926  0.063  0.012  0.000  0.000
caged_box_repaired          comp     0.978  0.020  0.002  0.000  0.001
                            solo     0.927  0.063  0.011  0.000  0.000
caged_voiced                comp     0.979  0.018  0.002  0.000  0.001
                            solo     0.926  0.062  0.012  0.000  0.000
caged_voiced_repaired       comp     0.979  0.018  0.002  0.000  0.001
                            solo     0.927  0.062  0.011  0.000  0.000
combined_all_tuned          comp     0.978  0.017  0.003  0.001  0.001
                            solo     0.952  0.045  0.003  0.000  0.000
combined_all_tuned_repaired comp     0.978  0.017  0.003  0.001  0.001
                            solo     0.954  0.045  0.001  0.000  0.000

## ASCII Tab Comparison: Real Audio vs Ground-Truth MIDI

This section integrates the `AlgoToASCII` rendering logic into the end-to-end audio pipeline.

After the audio eval cell creates `audio_predictions_df`, `audio_matches_df`, and `audio_metrics_df`, this cell renders:

- **Real audio → Basic Pitch → algorithm tab**
- **GuitarSet ground-truth MIDI/string/fret tab**
- optional **matched-note-only** tabs for an apples-to-apples view of notes that Basic Pitch matched by pitch and onset

The output is printed in the notebook and saved under `AUDIO_OUTPUT_DIR / 'ascii_tabs'`.

In [29]:

# ============================================================
# ASCII tab rendering for real-audio predictions vs GuitarSet ground truth
# ============================================================
# This integrates the standalone AlgoToASCII notebook into the end-to-end
# audio evaluation notebook.
#
# It renders:
#   1) REAL AUDIO → Basic Pitch → algorithm string/fret predictions
#   2) GUITARSET GROUND-TRUTH MIDI → true string/fret positions
#
# The two tabs use the same beat/tempo grid from the JAMS file, so you can
# visually inspect how audio note-detection errors change the final tab.

DISPLAY_TO_STRING = [5, 4, 3, 2, 1, 0]   # high E on top, low E on bottom
STRING_LABELS     = ['e', 'B', 'G', 'D', 'A', 'E']

def _clean_int_or_none(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass
    return int(round(float(value)))

def _clean_float_or_default(value, default=0.0):
    if value is None:
        return default
    try:
        if pd.isna(value):
            return default
    except TypeError:
        pass
    return float(value)

def _build_time_grid(beats, tempo, end_time, subdivisions_per_beat=2):
    """Build a quantized render grid from JAMS beats when available."""
    beats = sorted([float(b) for b in (beats or []) if b is not None])

    if len(beats) >= 2:
        grid = []
        for i in range(len(beats) - 1):
            step = (beats[i + 1] - beats[i]) / subdivisions_per_beat
            if step <= 0:
                continue
            for j in range(subdivisions_per_beat):
                grid.append(beats[i] + j * step)

        if grid:
            last_step = (beats[-1] - beats[-2]) / subdivisions_per_beat
            if last_step <= 0:
                last_step = 0.25
            while grid[-1] < end_time:
                grid.append(grid[-1] + last_step)
            return grid

    if tempo and float(tempo) > 0:
        step = 60.0 / float(tempo) / subdivisions_per_beat
        n = int(end_time / step) + subdivisions_per_beat + 2
        return [i * step for i in range(n)]

    # Last fallback: quarter-second grid.
    return [i * 0.25 for i in range(int(end_time / 0.25) + 3)]

def render_ascii_tab(parsed, subdivisions_per_beat=2, beats_per_measure=4,
                     measures_per_line=4, col_width=3, max_notes=None):
    """Render a dict with notes/beats/tempo into six-line ASCII guitar tab."""
    notes = sorted(parsed.get('notes', []), key=lambda n: (n.get('start', 0.0), n.get('midi', 0)))
    if max_notes is not None:
        notes = notes[:max_notes]
    if not notes:
        return '(no notes)'

    end_time = max(
        _clean_float_or_default(n.get('start'), 0.0) + _clean_float_or_default(n.get('duration'), 0.5)
        for n in notes
    ) + 0.5
    grid = _build_time_grid(
        parsed.get('beats', []),
        parsed.get('tempo'),
        end_time,
        subdivisions_per_beat=subdivisions_per_beat,
    )

    n_cols = len(grid)
    cells = [[None] * n_cols for _ in range(6)]
    collisions = 0
    skipped = 0

    for note in notes:
        string = _clean_int_or_none(note.get('string'))
        fret = _clean_int_or_none(note.get('fret'))
        if string is None or fret is None or string not in DISPLAY_TO_STRING:
            skipped += 1
            continue
        col = min(range(n_cols), key=lambda i: abs(grid[i] - _clean_float_or_default(note.get('start'), 0.0)))
        row = DISPLAY_TO_STRING.index(string)
        if cells[row][col] is not None:
            collisions += 1
        cells[row][col] = fret

    def fmt(v):
        if v is None:
            return '-' * col_width
        s = str(v)
        return s[:col_width] if len(s) >= col_width else s + '-' * (col_width - len(s))

    formatted = [[fmt(cells[r][c]) for c in range(n_cols)] for r in range(6)]
    cols_per_measure = beats_per_measure * subdivisions_per_beat
    cols_per_line = cols_per_measure * measures_per_line

    lines = []
    title = parsed.get('title')
    if title:
        lines.append(str(title))
        lines.append('-' * min(len(str(title)), 80))

    for start in range(0, n_cols, cols_per_line):
        end = min(start + cols_per_line, n_cols)
        for row in range(6):
            parts = []
            for c in range(start, end):
                if c > start and (c - start) % cols_per_measure == 0:
                    parts.append('|')
                parts.append(formatted[row][c])
            lines.append(f"{STRING_LABELS[row]}|{''.join(parts)}|")
        lines.append('')

    if collisions:
        lines.append(f"[note: {collisions} same-grid-cell collisions; later note shown]")
    if skipped:
        lines.append(f"[note: {skipped} notes skipped because string/fret was missing or invalid]")
    return '\n'.join(lines)

def _record_lookup_from_pairs(record_audio_pairs):
    """Map recording_id -> parsed GuitarSet record from records_to_run/paired_records."""
    lookup = {}
    for item in record_audio_pairs:
        if isinstance(item, tuple):
            rec = item[0]
        else:
            rec = item
        lookup[rec['recording']] = rec
    return lookup

def ground_truth_record_to_render_dict(record, title=None):
    """Render-ready dict for GuitarSet ground-truth MIDI/string/fret notes."""
    notes = []
    for n in record.get('notes', []):
        string = _clean_int_or_none(n.get('true_string'))
        fret = _clean_int_or_none(n.get('true_fret'))
        if string is None or fret is None or not (0 <= fret <= MAX_FRET):
            continue
        notes.append({
            'start': _clean_float_or_default(n.get('start'), 0.0),
            'duration': _clean_float_or_default(n.get('duration'), 0.5),
            'midi': _clean_int_or_none(n.get('midi')),
            'string': string,
            'fret': fret,
        })

    return {
        'recording': record.get('recording'),
        'title': title or f"GROUND TRUTH MIDI TAB — {record.get('recording')}",
        'notes': notes,
        'beats': record.get('beats', []),
        'tempo': record.get('tempo'),
        'key': record.get('key'),
    }

def audio_predictions_to_render_dict(audio_predictions_df, recording_id, method='combined_all_tuned',
                                     record_lookup=None, title=None):
    """Render-ready dict for real-audio Basic Pitch → algorithm predictions."""
    if audio_predictions_df is None or audio_predictions_df.empty:
        raise ValueError('audio_predictions_df is empty. Run the audio evaluation cell first.')

    rec_df = audio_predictions_df[
        (audio_predictions_df['recording'] == recording_id) &
        (audio_predictions_df['method'] == method)
    ].sort_values(['start', 'midi'])

    if rec_df.empty:
        raise ValueError(f'No audio predictions found for recording={recording_id}, method={method}')

    notes = []
    for _, row in rec_df.iterrows():
        string = _clean_int_or_none(row.get('pred_string'))
        fret = _clean_int_or_none(row.get('pred_fret'))
        if string is None or fret is None or not (0 <= fret <= MAX_FRET):
            continue
        notes.append({
            'start': _clean_float_or_default(row.get('start'), 0.0),
            'duration': _clean_float_or_default(row.get('duration'), 0.5),
            'midi': _clean_int_or_none(row.get('midi')),
            'string': string,
            'fret': fret,
            'amplitude': row.get('amplitude'),
        })

    rec_meta = (record_lookup or {}).get(recording_id, {})
    return {
        'recording': recording_id,
        'title': title or f"REAL AUDIO PREDICTED TAB — {recording_id} — {method}",
        'notes': notes,
        'beats': rec_meta.get('beats', []),
        'tempo': rec_meta.get('tempo'),
        'key': rec_meta.get('key'),
    }

def matched_notes_to_render_dict(audio_matches_df, recording_id, method='combined_all_tuned',
                                 source='pred', record_lookup=None, title=None):
    """Optional apples-to-apples render for notes that matched by pitch+onset only."""
    if source not in ('pred', 'truth'):
        raise ValueError("source must be 'pred' or 'truth'")
    if audio_matches_df is None or audio_matches_df.empty:
        raise ValueError('audio_matches_df is empty. Run the audio evaluation cell first.')

    rec_df = audio_matches_df[
        (audio_matches_df['recording'] == recording_id) &
        (audio_matches_df['method'] == method)
    ].sort_values(['gt_start' if source == 'truth' else 'pred_start', 'midi'])

    if source == 'pred':
        start_col, string_col, fret_col = 'pred_start', 'pred_string', 'pred_fret'
        default_title = f"MATCHED ONLY: REAL AUDIO PREDICTED TAB — {recording_id} — {method}"
    else:
        start_col, string_col, fret_col = 'gt_start', 'true_string', 'true_fret'
        default_title = f"MATCHED ONLY: GROUND TRUTH TAB — {recording_id} — {method}"

    notes = []
    for _, row in rec_df.iterrows():
        string = _clean_int_or_none(row.get(string_col))
        fret = _clean_int_or_none(row.get(fret_col))
        if string is None or fret is None or not (0 <= fret <= MAX_FRET):
            continue
        notes.append({
            'start': _clean_float_or_default(row.get(start_col), 0.0),
            'duration': 0.5,
            'midi': _clean_int_or_none(row.get('midi')),
            'string': string,
            'fret': fret,
        })

    rec_meta = (record_lookup or {}).get(recording_id, {})
    return {
        'recording': recording_id,
        'title': title or default_title,
        'notes': notes,
        'beats': rec_meta.get('beats', []),
        'tempo': rec_meta.get('tempo'),
        'key': rec_meta.get('key'),
    }

def render_audio_vs_ground_truth(recording_id=None, method='combined_all_tuned', max_notes=160,
                                 save_txt=True, show_matched_only=True):
    """Print and optionally save side-by-side ASCII tabs for one recording."""
    if 'records_to_run' in globals():
        record_lookup = _record_lookup_from_pairs(records_to_run)
    elif 'paired_records' in globals():
        record_lookup = _record_lookup_from_pairs(paired_records)
    else:
        record_lookup = {r['recording']: r for r in records}

    if recording_id is None:
        if 'audio_predictions_df' not in globals() or audio_predictions_df.empty:
            raise ValueError('No recording_id supplied and audio_predictions_df is empty.')
        recording_id = sorted(audio_predictions_df['recording'].unique())[0]

    if recording_id not in record_lookup:
        # Fall back to all parsed records in case records_to_run was limited.
        all_records_lookup = {r['recording']: r for r in records}
        if recording_id in all_records_lookup:
            record_lookup[recording_id] = all_records_lookup[recording_id]
        else:
            raise ValueError(f'{recording_id} not found in records_to_run, paired_records, or records.')

    pred_dict = audio_predictions_to_render_dict(
        audio_predictions_df, recording_id, method=method, record_lookup=record_lookup
    )
    gt_dict = ground_truth_record_to_render_dict(record_lookup[recording_id])

    print('=' * 88)
    print(f'Recording: {recording_id}')
    print(f'Method:    {method}')
    if 'audio_metrics_df' in globals() and not audio_metrics_df.empty:
        metric_row = audio_metrics_df[
            (audio_metrics_df['recording'] == recording_id) &
            (audio_metrics_df['method'] == method)
        ]
        if not metric_row.empty:
            cols = [
                'n_gt_notes', 'n_basic_pitch_notes', 'n_pitch_onset_matches',
                'pitch_precision', 'pitch_recall', 'exact_tab_f1',
                'tab_accuracy_given_pitch_match'
            ]
            display(metric_row[[c for c in cols if c in metric_row.columns]].round(4))

    pred_tab = render_ascii_tab(pred_dict, max_notes=max_notes)
    gt_tab = render_ascii_tab(gt_dict, max_notes=max_notes)

    print('\n' + '=' * 88)
    print('REAL AUDIO → BASIC PITCH → ALGORITHM TAB')
    print('=' * 88)
    print(pred_tab)

    print('\n' + '=' * 88)
    print('GUITARSET GROUND-TRUTH MIDI TAB')
    print('=' * 88)
    print(gt_tab)

    outputs = {}
    if save_txt:
        ascii_dir = AUDIO_OUTPUT_DIR / 'ascii_tabs'
        ascii_dir.mkdir(parents=True, exist_ok=True)

        safe_recording = re.sub(r'[^A-Za-z0-9_.-]+', '_', recording_id)
        safe_method = re.sub(r'[^A-Za-z0-9_.-]+', '_', method)

        pred_path = ascii_dir / f'{safe_recording}__{safe_method}__real_audio_predicted_tab.txt'
        gt_path = ascii_dir / f'{safe_recording}__ground_truth_midi_tab.txt'
        pred_path.write_text(pred_tab)
        gt_path.write_text(gt_tab)
        outputs['real_audio_predicted_tab'] = pred_path
        outputs['ground_truth_midi_tab'] = gt_path

    if show_matched_only and 'audio_matches_df' in globals() and not audio_matches_df.empty:
        matched_pred = matched_notes_to_render_dict(
            audio_matches_df, recording_id, method=method, source='pred', record_lookup=record_lookup
        )
        matched_truth = matched_notes_to_render_dict(
            audio_matches_df, recording_id, method=method, source='truth', record_lookup=record_lookup
        )
        matched_pred_tab = render_ascii_tab(matched_pred, max_notes=max_notes)
        matched_truth_tab = render_ascii_tab(matched_truth, max_notes=max_notes)

        print('\n' + '=' * 88)
        print('MATCHED-NOTE-ONLY VIEW: REAL AUDIO PREDICTED')
        print('=' * 88)
        print(matched_pred_tab)

        print('\n' + '=' * 88)
        print('MATCHED-NOTE-ONLY VIEW: GROUND TRUTH')
        print('=' * 88)
        print(matched_truth_tab)

        if save_txt:
            matched_pred_path = ascii_dir / f'{safe_recording}__{safe_method}__matched_only_real_audio_predicted_tab.txt'
            matched_gt_path = ascii_dir / f'{safe_recording}__{safe_method}__matched_only_ground_truth_tab.txt'
            matched_pred_path.write_text(matched_pred_tab)
            matched_gt_path.write_text(matched_truth_tab)
            outputs['matched_only_real_audio_predicted_tab'] = matched_pred_path
            outputs['matched_only_ground_truth_tab'] = matched_gt_path

    if outputs:
        print('\nSaved ASCII tabs:')
        for label, path in outputs.items():
            print(f' - {label}: {path.resolve()}')

    return outputs

def export_ascii_tabs_for_all_audio_runs(method='combined_all_tuned', max_notes=None):
    """Batch export predicted-vs-GT ASCII tabs for every successful audio run."""
    if 'audio_predictions_df' not in globals() or audio_predictions_df.empty:
        raise ValueError('audio_predictions_df is empty. Run the audio evaluation cell first.')

    ascii_dir = AUDIO_OUTPUT_DIR / 'ascii_tabs'
    ascii_dir.mkdir(parents=True, exist_ok=True)

    exported = []
    for recording_id in sorted(audio_predictions_df.loc[audio_predictions_df['method'] == method, 'recording'].unique()):
        outputs = render_audio_vs_ground_truth(
            recording_id=recording_id,
            method=method,
            max_notes=max_notes,
            save_txt=True,
            show_matched_only=False,
        )
        exported.extend([str(p) for p in outputs.values()])

    exported_index = ascii_dir / f'exported_ascii_tabs__{method}.txt'
    exported_index.write_text('\n'.join(exported))
    print(f'\nExported {len(exported)} ASCII tab files.')
    print('Index:', exported_index.resolve())
    return exported

# Demo: render the first successful audio eval recording.
# Change AUDIO_TAB_RECORDING to any recording_id in audio_predictions_df['recording'].
AUDIO_TAB_METHOD = 'combined_all_tuned'
AUDIO_TAB_RECORDING = (
    sorted(audio_predictions_df['recording'].unique())[0]
    if 'audio_predictions_df' in globals() and not audio_predictions_df.empty
    else None
)

if AUDIO_TAB_RECORDING is not None:
    render_audio_vs_ground_truth(
        recording_id=AUDIO_TAB_RECORDING,
        method=AUDIO_TAB_METHOD,
        max_notes=160,
        save_txt=True,
        show_matched_only=True,
    )
else:
    print('Run the audio evaluation cell first, then rerun this cell to render ASCII tabs.')


Recording: 00_Funk1-114-Ab_solo
Method:    combined_all_tuned


,n_gt_notes,n_basic_pitch_notes,n_pitch_onset_matches,pitch_precision,pitch_recall,exact_tab_f1,tab_accuracy_given_pitch_match
284,58,69,47,0.6812,0.8103,0.5197,0.7021



REAL AUDIO → BASIC PITCH → ALGORITHM TAB
REAL AUDIO PREDICTED TAB — 00_Funk1-114-Ab_solo — combined_all_tuned
--------------------------------------------------------------------
e|------------------------|------------------------|------------------------|------------------------|
B|4-----4-----------------|------------------------|------------------------|------------------------|
G|------------4--4--1-----|------4--4--------------|------------4-----------|------------------------|
D|------------------------|---6-----------3--4--3--|---------6--------1-----|---3--4--6--8--9--8--6--|
A|6-----------2--3--6-----|------------6-----------|6-----------2--3--------|------------------------|
E|---------4--------4-----|---4--7--7--------------|---------4--------------|------------------------|

e|------------------------|------4-----------------|------------------------|------------------------|
B|------------------------|------------------------|------------------------|---------------------


## 15b. Repair Diff + Whole-Measure Context

Run this after the audio evaluation and ASCII-rendering cells. It prints every repaired note as an old-vs-new string/fret diff, then renders the **entire measure** around each repair:

- original method tab for that measure
- repaired method tab for that measure
- GuitarSet ground-truth tab for that measure, when available
- a note table for every detected note in the measure

This is the easiest way to see whether `isolated_position_outlier` or another repair reason actually makes sense in musical context.


In [30]:

# ============================================================
# Repair diff + whole-measure context visualization
# ============================================================
# This cell explains what repair changed and shows the whole measure around each edit.
# It depends on:
#   - audio_predictions_df from the audio evaluation cell
#   - render_ascii_tab from the ASCII rendering cell

import math
from IPython.display import display

REPAIR_CONTEXT_METHODS = ['combined_all_tuned_repaired', 'caged_voiced_repaired', 'caged_box_repaired']
REPAIR_CONTEXT_MAX_EXAMPLES = 12
REPAIR_CONTEXT_BEATS_PER_MEASURE = 4
REPAIR_CONTEXT_SUBDIVISIONS_PER_BEAT = 2
REPAIR_CONTEXT_SAVE_TXT = True


def _get_record_lookup_for_context():
    if 'records_to_run' in globals():
        lookup = _record_lookup_from_pairs(records_to_run)
    elif 'paired_records' in globals():
        lookup = _record_lookup_from_pairs(paired_records)
    else:
        lookup = {}
    if 'records' in globals():
        for r in records:
            lookup.setdefault(r['recording'], r)
    return lookup


def build_repair_diff_table(predictions_df=None, methods=None):
    """Return one row per repaired note, including old and new string/fret."""
    if predictions_df is None:
        predictions_df = globals().get('audio_predictions_df')
    if predictions_df is None or predictions_df.empty:
        print('audio_predictions_df is empty. Run the audio evaluation cell first.')
        return pd.DataFrame()

    df = predictions_df.copy()
    if 'repair_changed_note' not in df.columns:
        print('No repair_changed_note column found. Run a repaired method first.')
        return pd.DataFrame()

    repaired = df[df['method'].astype(str).str.endswith('_repaired')].copy()
    repaired = repaired[repaired['repair_changed_note'].fillna(False).astype(bool)]
    if methods is not None:
        repaired = repaired[repaired['method'].isin(methods)]

    rows = []
    for _, r in repaired.sort_values(['recording', 'method', 'start', 'midi']).iterrows():
        repaired_method = str(r['method'])
        base_method = repaired_method.replace('_repaired', '')

        # Prefer the repair_old_* columns baked into the strict repair layer.
        old_string = r.get('repair_old_string', np.nan)
        old_fret = r.get('repair_old_fret', np.nan)
        new_string = r.get('repair_new_string', r.get('pred_string', np.nan))
        new_fret = r.get('repair_new_fret', r.get('pred_fret', np.nan))

        # Backfill old position from the base method when needed.
        if pd.isna(old_string) or pd.isna(old_fret):
            candidates = df[
                (df['method'] == base_method) &
                (df['recording'] == r['recording']) &
                (np.isclose(df['start'].astype(float), float(r['start']), atol=1e-6)) &
                (df['midi'].astype(int) == int(r['midi']))
            ]
            if not candidates.empty:
                old = candidates.iloc[0]
                old_string = old.get('pred_string')
                old_fret = old.get('pred_fret')

        rows.append({
            'recording': r.get('recording'),
            'base_method': base_method,
            'repaired_method': repaired_method,
            'start': float(r.get('start', 0.0)),
            'duration': float(r.get('duration', 0.0)),
            'midi': int(r.get('midi')) if not pd.isna(r.get('midi')) else None,
            'note_name': r.get('note_name'),
            'old_string': old_string,
            'old_fret': old_fret,
            'new_string': new_string,
            'new_fret': new_fret,
            'reason': r.get('repair_reasons') or r.get('repair_report_reasons'),
            'repair_mode': r.get('repair_mode'),
            'repair_num_notes_changed': r.get('repair_num_notes_changed'),
            'repair_num_groups_changed': r.get('repair_num_groups_changed'),
        })

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.drop_duplicates(subset=['recording', 'repaired_method', 'start', 'midi', 'old_string', 'old_fret', 'new_string', 'new_fret'])
    return out


def _estimate_beat_length(record):
    beats = sorted([float(b) for b in record.get('beats', []) if b is not None])
    if len(beats) >= 2:
        diffs = np.diff(beats)
        diffs = diffs[diffs > 0]
        if len(diffs):
            return float(np.median(diffs))
    tempo = record.get('tempo')
    if tempo and float(tempo) > 0:
        return 60.0 / float(tempo)
    return 0.5


def measure_bounds_for_time(record, start_time, beats_per_measure=4):
    """Return measure_index, measure_start, measure_end for a note start time."""
    beats = sorted([float(b) for b in record.get('beats', []) if b is not None])
    beat_len = _estimate_beat_length(record)

    if len(beats) >= 2:
        beat_idx = int(np.searchsorted(beats, float(start_time), side='right') - 1)
        beat_idx = max(0, beat_idx)
        measure_idx = beat_idx // beats_per_measure
        start_beat_idx = measure_idx * beats_per_measure
        end_beat_idx = start_beat_idx + beats_per_measure
        measure_start = beats[start_beat_idx] if start_beat_idx < len(beats) else beats[-1]
        if end_beat_idx < len(beats):
            measure_end = beats[end_beat_idx]
        else:
            measure_end = measure_start + beats_per_measure * beat_len
        return measure_idx + 1, float(measure_start), float(measure_end)

    measure_len = beats_per_measure * beat_len
    measure_idx = int(math.floor(float(start_time) / measure_len)) if measure_len > 0 else 0
    measure_start = measure_idx * measure_len
    measure_end = measure_start + measure_len
    return measure_idx + 1, float(measure_start), float(measure_end)


def _predictions_measure_render_dict(predictions_df, recording_id, method, measure_start, measure_end, record_lookup, title):
    rec_df = predictions_df[
        (predictions_df['recording'] == recording_id) &
        (predictions_df['method'] == method) &
        (predictions_df['start'].astype(float) >= measure_start) &
        (predictions_df['start'].astype(float) < measure_end)
    ].sort_values(['start', 'midi'])

    notes = []
    for _, row in rec_df.iterrows():
        string = _clean_int_or_none(row.get('pred_string'))
        fret = _clean_int_or_none(row.get('pred_fret'))
        if string is None or fret is None or not (0 <= fret <= MAX_FRET):
            continue
        notes.append({
            'start': _clean_float_or_default(row.get('start'), 0.0),
            'duration': _clean_float_or_default(row.get('duration'), 0.5),
            'midi': _clean_int_or_none(row.get('midi')),
            'string': string,
            'fret': fret,
        })

    record = record_lookup.get(recording_id, {})
    beats = [b for b in record.get('beats', []) if measure_start - 1e-9 <= float(b) <= measure_end + 1e-9]
    return {
        'recording': recording_id,
        'title': title,
        'notes': notes,
        'beats': beats,
        'tempo': record.get('tempo'),
        'key': record.get('key'),
    }


def _ground_truth_measure_render_dict(record, measure_start, measure_end, title):
    notes = []
    for n in record.get('notes', []):
        st = _clean_float_or_default(n.get('start'), 0.0)
        if st < measure_start or st >= measure_end:
            continue
        string = _clean_int_or_none(n.get('true_string'))
        fret = _clean_int_or_none(n.get('true_fret'))
        if string is None or fret is None or not (0 <= fret <= MAX_FRET):
            continue
        notes.append({
            'start': st,
            'duration': _clean_float_or_default(n.get('duration'), 0.5),
            'midi': _clean_int_or_none(n.get('midi')),
            'string': string,
            'fret': fret,
        })
    beats = [b for b in record.get('beats', []) if measure_start - 1e-9 <= float(b) <= measure_end + 1e-9]
    return {
        'recording': record.get('recording'),
        'title': title,
        'notes': notes,
        'beats': beats,
        'tempo': record.get('tempo'),
        'key': record.get('key'),
    }


def _measure_note_table(predictions_df, recording_id, base_method, repaired_method, measure_start, measure_end):
    base_df = predictions_df[
        (predictions_df['recording'] == recording_id) &
        (predictions_df['method'] == base_method) &
        (predictions_df['start'].astype(float) >= measure_start) &
        (predictions_df['start'].astype(float) < measure_end)
    ].copy()
    rep_df = predictions_df[
        (predictions_df['recording'] == recording_id) &
        (predictions_df['method'] == repaired_method) &
        (predictions_df['start'].astype(float) >= measure_start) &
        (predictions_df['start'].astype(float) < measure_end)
    ].copy()

    if base_df.empty and rep_df.empty:
        return pd.DataFrame()

    key_cols = ['start', 'midi']
    base_small = base_df[key_cols + [c for c in ['duration', 'note_name', 'pred_string', 'pred_fret'] if c in base_df.columns]].copy()
    rep_small = rep_df[key_cols + [c for c in ['pred_string', 'pred_fret', 'repair_changed_note', 'repair_reasons'] if c in rep_df.columns]].copy()
    base_small = base_small.rename(columns={'pred_string': 'old_string', 'pred_fret': 'old_fret'})
    rep_small = rep_small.rename(columns={'pred_string': 'new_string', 'pred_fret': 'new_fret'})

    merged = pd.merge(base_small, rep_small, on=key_cols, how='outer')
    merged = merged.sort_values(['start', 'midi'])
    merged['measure_time'] = (merged['start'].astype(float) - measure_start).round(3)
    merged['changed'] = (merged['old_string'] != merged['new_string']) | (merged['old_fret'] != merged['new_fret'])

    cols = ['measure_time', 'start', 'duration', 'midi', 'note_name', 'old_string', 'old_fret', 'new_string', 'new_fret', 'changed', 'repair_reasons']
    return merged[[c for c in cols if c in merged.columns]]


def print_repair_measure_context(diff_row, predictions_df=None, save_txt=True):
    """Print old vs repaired measure context for one diff row."""
    if predictions_df is None:
        predictions_df = globals().get('audio_predictions_df')
    record_lookup = _get_record_lookup_for_context()

    recording_id = diff_row['recording']
    base_method = diff_row['base_method']
    repaired_method = diff_row['repaired_method']
    start_time = float(diff_row['start'])

    record = record_lookup.get(recording_id)
    if record is None:
        print(f'Could not find record metadata for {recording_id}.')
        return None

    measure_idx, measure_start, measure_end = measure_bounds_for_time(
        record, start_time, beats_per_measure=REPAIR_CONTEXT_BEATS_PER_MEASURE
    )

    header = (
        f"{recording_id} | {base_method} → {repaired_method} | "
        f"measure {measure_idx} ({measure_start:.3f}s–{measure_end:.3f}s) | "
        f"changed note {diff_row.get('note_name')} midi={diff_row.get('midi')} at {start_time:.3f}s | "
        f"{diff_row.get('old_string')}/{diff_row.get('old_fret')} → {diff_row.get('new_string')}/{diff_row.get('new_fret')} | "
        f"reason={diff_row.get('reason')}"
    )

    base_dict = _predictions_measure_render_dict(
        predictions_df, recording_id, base_method, measure_start, measure_end, record_lookup,
        title=f'ORIGINAL — {base_method} — measure {measure_idx}'
    )
    repaired_dict = _predictions_measure_render_dict(
        predictions_df, recording_id, repaired_method, measure_start, measure_end, record_lookup,
        title=f'REPAIRED — {repaired_method} — measure {measure_idx}'
    )
    truth_dict = _ground_truth_measure_render_dict(
        record, measure_start, measure_end,
        title=f'GROUND TRUTH — measure {measure_idx}'
    )

    base_tab = render_ascii_tab(
        base_dict,
        subdivisions_per_beat=REPAIR_CONTEXT_SUBDIVISIONS_PER_BEAT,
        beats_per_measure=REPAIR_CONTEXT_BEATS_PER_MEASURE,
        measures_per_line=1,
    )
    repaired_tab = render_ascii_tab(
        repaired_dict,
        subdivisions_per_beat=REPAIR_CONTEXT_SUBDIVISIONS_PER_BEAT,
        beats_per_measure=REPAIR_CONTEXT_BEATS_PER_MEASURE,
        measures_per_line=1,
    )
    truth_tab = render_ascii_tab(
        truth_dict,
        subdivisions_per_beat=REPAIR_CONTEXT_SUBDIVISIONS_PER_BEAT,
        beats_per_measure=REPAIR_CONTEXT_BEATS_PER_MEASURE,
        measures_per_line=1,
    )
    note_table = _measure_note_table(predictions_df, recording_id, base_method, repaired_method, measure_start, measure_end)

    print('\n' + '=' * 110)
    print(header)
    print('=' * 110)
    print('\nNOTE TABLE FOR THIS MEASURE')
    display(note_table)
    print('\nORIGINAL MEASURE')
    print(base_tab)
    print('\nREPAIRED MEASURE')
    print(repaired_tab)
    print('\nGROUND TRUTH MEASURE')
    print(truth_tab)

    if save_txt:
        context_dir = AUDIO_OUTPUT_DIR / 'repair_measure_contexts'
        context_dir.mkdir(parents=True, exist_ok=True)
        safe_recording = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(recording_id))
        safe_method = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(repaired_method))
        safe_start = f'{start_time:.3f}'.replace('.', 'p')
        out_path = context_dir / f'{safe_recording}__{safe_method}__measure_{measure_idx}__t_{safe_start}.txt'
        text = '\n'.join([
            header,
            '',
            'NOTE TABLE FOR THIS MEASURE',
            note_table.to_string(index=False) if not note_table.empty else '(no notes)',
            '',
            'ORIGINAL MEASURE',
            base_tab,
            '',
            'REPAIRED MEASURE',
            repaired_tab,
            '',
            'GROUND TRUTH MEASURE',
            truth_tab,
        ])
        out_path.write_text(text)
        print('Saved context:', out_path.resolve())
        return out_path
    return None


# Auto-run summary and contexts after Run all.
repair_diffs_df = build_repair_diff_table(methods=REPAIR_CONTEXT_METHODS)

if repair_diffs_df.empty:
    print('No repaired notes found for the selected methods.')
else:
    print(f'Found {len(repair_diffs_df)} repaired note(s). Full diff table:')
    display(repair_diffs_df)

    saved_contexts = []
    for _, diff_row in repair_diffs_df.head(REPAIR_CONTEXT_MAX_EXAMPLES).iterrows():
        out = print_repair_measure_context(diff_row, save_txt=REPAIR_CONTEXT_SAVE_TXT)
        if out is not None:
            saved_contexts.append(str(out))

    if saved_contexts:
        print('\nSaved repair measure context files:')
        for p in saved_contexts:
            print(' -', p)


Found 4 repaired note(s). Full diff table:


,recording,base_method,repaired_method,start,duration,midi,note_name,old_string,old_fret,new_string,new_fret,reason,repair_mode,repair_num_notes_changed,repair_num_groups_changed
0,00_Funk1-97-C_solo,combined_all_tuned,combined_all_tuned_repaired,10.803698,1.870490,57,A3,0.0,17.0,2.0,7.0,isolated_position_outlier,strict_conservative,1.0,1.0
1,00_Funk2-108-Eb_solo,caged_box,caged_box_repaired,17.866416,0.198654,58,A#3,0.0,18.0,2.0,8.0,isolated_position_outlier,strict_conservative,1.0,1.0
2,00_Funk2-108-Eb_solo,caged_voiced,caged_voiced_repaired,17.866416,0.198654,58,A#3,0.0,18.0,2.0,8.0,isolated_position_outlier,strict_conservative,1.0,1.0
3,00_Funk2-108-Eb_solo,combined_all_tuned,combined_all_tuned_repaired,17.866416,0.198654,58,A#3,0.0,18.0,2.0,8.0,isolated_position_outlier,strict_conservative,1.0,1.0



00_Funk1-97-C_solo | combined_all_tuned → combined_all_tuned_repaired | measure 5 (9.897s–12.371s) | changed note A3 midi=57 at 10.804s | 0.0/17.0 → 2.0/7.0 | reason=isolated_position_outlier

NOTE TABLE FOR THIS MEASURE


,measure_time,start,duration,midi,note_name,old_string,old_fret,new_string,new_fret,changed,repair_reasons
0,0.035,9.931666,0.291533,45,A2,0,5,0,5,False,
1,0.326,10.223200,0.441179,45,A2,0,5,0,5,False,
2,0.326,10.223200,0.301859,50,D3,2,0,2,0,False,
3,0.640,10.536669,2.079470,48,C3,1,3,1,3,False,
4,0.640,10.536669,0.267029,57,A3,2,7,2,7,False,
5,0.907,10.803698,1.870490,57,A3,0,17,2,7,True,isolated_position_outlier
6,0.907,10.803698,0.290249,85,C#6,5,21,5,21,False,isolated_position_outlier



ORIGINAL MEASURE
ORIGINAL — combined_all_tuned — measure 5
-----------------------------------------
e|---------21-------------|
B|------------------------|
G|------------------------|
D|---0--7-----------------|
A|------3-----------------|
E|5--5-----17-------------|

e|------------|
B|------------|
G|------------|
D|------------|
A|------------|
E|------------|


REPAIRED MEASURE
REPAIRED — combined_all_tuned_repaired — measure 5
--------------------------------------------------
e|---------21-------------|
B|------------------------|
G|------------------------|
D|---0--7--7--------------|
A|------3-----------------|
E|5--5--------------------|

e|------------|
B|------------|
G|------------|
D|------------|
A|------------|
E|------------|


GROUND TRUTH MEASURE
GROUND TRUTH — measure 5
------------------------
e|------------------------|
B|------------------------|
G|------2--2--------------|
D|------------------------|
A|---0--3-----------------|
E|------------------------|

e|---

...


ORIGINAL MEASURE
ORIGINAL — caged_box — measure 9
--------------------------------
e|22----------------------|
B|------------------------|
G|---------------6--4-----|
D|---8--8-----8--------6--|
A|------------------------|
E|18----------------------|

e|---------|
B|---------|
G|---------|
D|---------|
A|---------|
E|---------|

[note: 1 same-grid-cell collisions; later note shown]

REPAIRED MEASURE
REPAIRED — caged_box_repaired — measure 9
-----------------------------------------
e|22----------------------|
B|------------------------|
G|---------------6--4-----|
D|8--8--8-----8--------6--|
A|------------------------|
E|------------------------|

e|---------|
B|---------|
G|---------|
D|---------|
A|---------|
E|---------|

[note: 1 same-grid-cell collisions; later note shown]

GROUND TRUTH MEASURE
GROUND TRUTH — measure 9
------------------------
e|------------------------|
B|------------------------|
G|3--3--3--3--3--6--4-----|
D|---------------------6--|
A|------------------------

,measure_time,start,duration,midi,note_name,old_string,old_fret,new_string,new_fret,changed,repair_reasons
0,0.077,17.854806,0.140604,86,D6,5,22,5,22,False,isolated_position_outlier
1,0.089,17.866416,0.198654,58,A#3,0,18,2,8,True,isolated_position_outlier
2,0.287,18.065070,0.185760,58,A#3,2,8,2,8,False,
3,0.473,18.250829,0.150930,58,A#3,2,8,2,8,False,
4,0.624,18.401759,0.383129,58,A#3,2,8,2,8,False,
5,1.007,18.784888,0.417959,58,A#3,2,8,2,8,False,
6,1.460,19.237678,0.139320,61,C#4,3,6,3,6,False,
7,1.680,19.458267,0.301859,59,B3,3,4,3,4,False,
8,1.971,19.748517,0.256703,56,G#3,2,6,2,6,False,



ORIGINAL MEASURE
ORIGINAL — caged_voiced — measure 9
-----------------------------------
e|22----------------------|
B|------------------------|
G|---------------6--4-----|
D|---8--8-----8--------6--|
A|------------------------|
E|18----------------------|

e|---------|
B|---------|
G|---------|
D|---------|
A|---------|
E|---------|

[note: 1 same-grid-cell collisions; later note shown]

REPAIRED MEASURE
REPAIRED — caged_voiced_repaired — measure 9
--------------------------------------------
e|22----------------------|
B|------------------------|
G|---------------6--4-----|
D|8--8--8-----8--------6--|
A|------------------------|
E|------------------------|

e|---------|
B|---------|
G|---------|
D|---------|
A|---------|
E|---------|

[note: 1 same-grid-cell collisions; later note shown]

GROUND TRUTH MEASURE
GROUND TRUTH — measure 9
------------------------
e|------------------------|
B|------------------------|
G|3--3--3--3--3--6--4-----|
D|---------------------6--|
A|------------

,measure_time,start,duration,midi,note_name,old_string,old_fret,new_string,new_fret,changed,repair_reasons
0,0.077,17.854806,0.140604,86,D6,5,22,5,22,False,isolated_position_outlier
1,0.089,17.866416,0.198654,58,A#3,0,18,2,8,True,isolated_position_outlier
2,0.287,18.065070,0.185760,58,A#3,2,8,2,8,False,
3,0.473,18.250829,0.150930,58,A#3,2,8,2,8,False,
4,0.624,18.401759,0.383129,58,A#3,2,8,2,8,False,
5,1.007,18.784888,0.417959,58,A#3,2,8,2,8,False,
6,1.460,19.237678,0.139320,61,C#4,3,6,3,6,False,
7,1.680,19.458267,0.301859,59,B3,3,4,3,4,False,
8,1.971,19.748517,0.256703,56,G#3,2,6,2,6,False,



ORIGINAL MEASURE
ORIGINAL — combined_all_tuned — measure 9
-----------------------------------------
e|22----------------------|
B|------------------------|
G|---------------6--4-----|
D|---8--8-----8--------6--|
A|------------------------|
E|18----------------------|

e|---------|
B|---------|
G|---------|
D|---------|
A|---------|
E|---------|

[note: 1 same-grid-cell collisions; later note shown]

REPAIRED MEASURE
REPAIRED — combined_all_tuned_repaired — measure 9
--------------------------------------------------
e|22----------------------|
B|------------------------|
G|---------------6--4-----|
D|8--8--8-----8--------6--|
A|------------------------|
E|------------------------|

e|---------|
B|---------|
G|---------|
D|---------|
A|---------|
E|---------|

[note: 1 same-grid-cell collisions; later note shown]

GROUND TRUTH MEASURE
GROUND TRUTH — measure 9
------------------------
e|------------------------|
B|------------------------|
G|3--3--3--3--3--6--4-----|
D|----------------

In [31]:
export_ascii_tabs_for_all_audio_runs(
    method='combined_all_tuned',
    max_notes=None
)

Recording: 00_Funk1-114-Ab_solo
Method:    combined_all_tuned


,n_gt_notes,n_basic_pitch_notes,n_pitch_onset_matches,pitch_precision,pitch_recall,exact_tab_f1,tab_accuracy_given_pitch_match
284,58,69,47,0.6812,0.8103,0.5197,0.7021



REAL AUDIO → BASIC PITCH → ALGORITHM TAB
REAL AUDIO PREDICTED TAB — 00_Funk1-114-Ab_solo — combined_all_tuned
--------------------------------------------------------------------
e|------------------------|------------------------|------------------------|------------------------|
B|4-----4-----------------|------------------------|------------------------|------------------------|
G|------------4--4--1-----|------4--4--------------|------------4-----------|------------------------|
D|------------------------|---6-----------3--4--3--|---------6--------1-----|---3--4--6--8--9--8--6--|
A|6-----------2--3--6-----|------------6-----------|6-----------2--3--------|------------------------|
E|---------4--------4-----|---4--7--7--------------|---------4--------------|------------------------|

e|------------------------|------4-----------------|------------------------|------------------------|
B|------------------------|------------------------|------------------------|---------------------

,n_gt_notes,n_basic_pitch_notes,n_pitch_onset_matches,pitch_precision,pitch_recall,exact_tab_f1,tab_accuracy_given_pitch_match
116,62,79,54,0.6835,0.871,0.4113,0.537



REAL AUDIO → BASIC PITCH → ALGORITHM TAB
REAL AUDIO PREDICTED TAB — 00_Funk1-97-C_solo — combined_all_tuned
------------------------------------------------------------------
e|------------------------|------------------------|------------------------|------------------------|
B|------------------------|------------------------|------------------------|------------------------|
G|------6--5--------------|------6--5--------------|------6--5-----5--3--3--|3-----------------------|
D|---------0--------------|---------0--------------|---------0-----0--------|------------------------|
A|---6--6-----------------|---6--6-----------------|---6--6-----------3-----|3-----------------------|
E|------------------------|------------------------|5--4--------------------|------------------------|

e|---------21-------------|---------21-------------|------------------------|------------------------|
B|------------------------|------------------------|------------------------|------------------------|

...

,n_gt_notes,n_basic_pitch_notes,n_pitch_onset_matches,pitch_precision,pitch_recall,exact_tab_f1,tab_accuracy_given_pitch_match
158,398,341,308,0.9032,0.7739,0.3545,0.4253



REAL AUDIO → BASIC PITCH → ALGORITHM TAB
REAL AUDIO PREDICTED TAB — 05_Rock3-117-Bb_comp — combined_all_tuned
--------------------------------------------------------------------
e|------------------------|------------------------|------------------------|------------------------|
B|3-----3--3--------3-----|------6-----------------|------------------------|------6-----------6-----|
G|3--3--3--3-----3--3-----|5--5--5--5--5--5--5-----|------3--------------0--|------7-----7-----------|
D|3--3--3--3--3--3--3--0--|3--3--3--3--3--3--3--5--|------5--5--5--5--5--0--|7--7--7--7--7--7--7--5--|
A|1--1--1--1--1--1--1-----|---------------------5--|------5--------5--5-----|5--5--5--5--5-----5--5--|
E|---------------------5--|---------------------5--|3-----3--3--3--3--3--5--|------------------------|

e|------------------------|------1-----6-----------|---------------3--------|------5-----------5-----|
B|------8-----------8-----|------3--3--3-----3-----|---4--4-----4--4--4--5--|6--6--6--6--6--6--6--

,n_gt_notes,n_basic_pitch_notes,n_pitch_onset_matches,pitch_precision,pitch_recall,exact_tab_f1,tab_accuracy_given_pitch_match
26,93,96,91,0.9479,0.9785,0.709,0.7363



REAL AUDIO → BASIC PITCH → ALGORITHM TAB
REAL AUDIO PREDICTED TAB — 05_SS2-88-F_solo — combined_all_tuned
----------------------------------------------------------------
e|---------6--8--9--6-----|------------------------|------------------------|------------------------|
B|---------6-----------6--|---------6--------------|------------9--11-9--8--|6-----------------6--9--|
G|------------------------|------------8-----------|---------8--------------|------------------------|
D|------------------------|------------------------|------------------------|------------------------|
A|------------------------|------------------------|------------------------|------------------------|
E|------------------------|------------------------|------------------------|------------------------|

e|------------------------|------------------------|------------------------|------------------5--8--|
B|8--------------8--------|------------------8--6--|6-----------------6-----|------------------6-----|
G|-

['/content/drive/MyDrive/Capstone/outputs/audio_to_tab_basic_pitch_heldout/ascii_tabs/00_Funk1-114-Ab_solo__combined_all_tuned__real_audio_predicted_tab.txt',
 '/content/drive/MyDrive/Capstone/outputs/audio_to_tab_basic_pitch_heldout/ascii_tabs/00_Funk1-114-Ab_solo__ground_truth_midi_tab.txt',
 '/content/drive/MyDrive/Capstone/outputs/audio_to_tab_basic_pitch_heldout/ascii_tabs/00_Funk1-97-C_solo__combined_all_tuned__real_audio_predicted_tab.txt',
 '/content/drive/MyDrive/Capstone/outputs/audio_to_tab_basic_pitch_heldout/ascii_tabs/00_Funk1-97-C_solo__ground_truth_midi_tab.txt',
 '/content/drive/MyDrive/Capstone/outputs/audio_to_tab_basic_pitch_heldout/ascii_tabs/00_Funk2-108-Eb_solo__combined_all_tuned__real_audio_predicted_tab.txt',
 '/content/drive/MyDrive/Capstone/outputs/audio_to_tab_basic_pitch_heldout/ascii_tabs/00_Funk2-108-Eb_solo__ground_truth_midi_tab.txt',
 '/content/drive/MyDrive/Capstone/outputs/audio_to_tab_basic_pitch_heldout/ascii_tabs/00_Jazz3-150-C_comp__combined_all


## Final Reporting Table

Use this table for the class update. It is held-out test only when `USE_HELDOUT_SPLIT=True`.


In [32]:
print('Final audio output folder:')
print(AUDIO_OUTPUT_DIR.resolve())

print('\nCSV files currently in audio output folder:')
for f in sorted(AUDIO_OUTPUT_DIR.glob('*.csv')):
    print(' -', f.name)

print('\nASCII tab files currently in audio output folder:')
ascii_dir = AUDIO_OUTPUT_DIR / 'ascii_tabs'
if ascii_dir.exists():
    for f in sorted(ascii_dir.glob('*.txt'))[:50]:
        print(' - ascii_tabs/' + f.name)
else:
    print(' - none yet; run the ASCII tab comparison cell first.')


if 'audio_summary_df' in globals() and not audio_summary_df.empty:
    report_cols = [
        'method', 'recordings', 'n_gt_notes', 'n_basic_pitch_notes',
        'pitch_precision', 'pitch_recall', 'pitch_f1',
        'exact_tab_precision', 'exact_tab_recall', 'exact_tab_f1',
        'tab_accuracy_given_pitch_match',
        'string_accuracy_given_pitch_match',
        'fret_accuracy_given_pitch_match',
        'mean_runtime_sec_per_recording'
    ]
    audio_report_table = audio_summary_df[report_cols].copy()
    display(audio_report_table.round(4))

    report_path = AUDIO_OUTPUT_DIR / 'audio_to_tab_report_table_heldout_test.csv'
    audio_report_table.to_csv(report_path, index=False)
    print('\nSaved report table to:', report_path.resolve())
else:
    print('Run the audio evaluation cell first to create audio_summary_df.')

print('\nInterpretation:')
print('The previous held-out result measured fretboard assignment using ground-truth GuitarSet notes.')
print('This result starts from raw audio, runs Basic Pitch first, then predicts string/fret positions.')
print('So exact_tab_f1 is the stricter end-to-end audio-to-tab metric.')


Final audio output folder:
/content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/audio_to_tab_basic_pitch_heldout

CSV files currently in audio output folder:
 - audio_to_tab_eval_by_recording.csv
 - audio_to_tab_eval_summary.csv
 - audio_to_tab_failed_runs.csv
 - audio_to_tab_matches.csv
 - audio_to_tab_predictions_all.csv
 - audio_to_tab_report_table_heldout_test.csv
 - repair_changed_notes.csv
 - repair_impact_summary.csv

ASCII tab files currently in audio output folder:
 - ascii_tabs/00_Funk1-114-Ab_solo__combined_all_tuned__matched_only_ground_truth_tab.txt
 - ascii_tabs/00_Funk1-114-Ab_solo__combined_all_tuned__matched_only_real_audio_predicted_tab.txt
 - ascii_tabs/00_Funk1-114-Ab_solo__combined_all_tuned__real_audio_predicted_tab.txt
 - ascii_tabs/00_Funk1-114-Ab_solo__ground_truth_midi_tab.txt
 - ascii_tabs/00_Funk1-97-C_solo__combined_all_tuned__real_audio_predicted_tab.txt
 - ascii_tabs/00_Funk1-97-C_solo__ground_truth_midi_tab.txt
 - asc

,method,recordings,n_gt_notes,n_basic_pitch_notes,pitch_precision,pitch_recall,pitch_f1,exact_tab_precision,exact_tab_recall,exact_tab_f1,tab_accuracy_given_pitch_match,string_accuracy_given_pitch_match,fret_accuracy_given_pitch_match,mean_runtime_sec_per_recording
2,caged_voiced,54,10207,9561,0.802,0.7512,0.7758,0.5549,0.5197,0.5367,0.6918,0.6918,0.6918,0.7144
3,caged_voiced_repaired,54,10207,9561,0.802,0.7512,0.7758,0.5549,0.5197,0.5367,0.6918,0.6918,0.6918,0.7032
0,caged_box,54,10207,9561,0.802,0.7512,0.7758,0.5497,0.5149,0.5318,0.6854,0.6854,0.6854,1.8474
1,caged_box_repaired,54,10207,9561,0.802,0.7512,0.7758,0.5497,0.5149,0.5318,0.6854,0.6854,0.6854,0.6744
4,combined_all_tuned,54,10207,9561,0.802,0.7512,0.7758,0.5191,0.4862,0.5021,0.6472,0.6472,0.6472,0.7060
5,combined_all_tuned_repaired,54,10207,9561,0.802,0.7512,0.7758,0.5191,0.4862,0.5021,0.6472,0.6472,0.6472,0.7205



Saved report table to: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/audio_to_tab_basic_pitch_heldout/audio_to_tab_report_table_heldout_test.csv

Interpretation:
The previous held-out result measured fretboard assignment using ground-truth GuitarSet notes.
This result starts from raw audio, runs Basic Pitch first, then predicts string/fret positions.
So exact_tab_f1 is the stricter end-to-end audio-to-tab metric.


In [33]:
import pandas as pd

def string_error_audit(matches_df, method=None):
    df = matches_df.copy()
    if method is not None and 'method' in df.columns:
        df = df[df['method'] == method]
    df = df.dropna(subset=['pred_string','pred_fret','true_string','true_fret'])
    for c in ['pred_string','pred_fret','true_string','true_fret']:
        df[c] = df[c].astype(int)
    n = len(df)
    correct = (df['pred_string']==df['true_string']) & (df['pred_fret']==df['true_fret'])
    wrong = df[~correct].copy()
    nw = len(wrong)
    wrong['string_diff'] = wrong['pred_string'] - wrong['true_string']
    wrong['fret_diff']   = wrong['pred_fret']   - wrong['true_fret']
    wrong['abs_string_diff'] = wrong['string_diff'].abs()

    print(f"Matched notes (detected + pitch-correct): {n}")
    print(f"  exact-correct placement : {n-nw}  ({(n-nw)/n:.1%})")
    print(f"  wrong placement         : {nw}  ({nw/n:.1%})\n")
    if nw == 0: return wrong
    print("Of the WRONG notes — how many strings off:")
    for k, v in wrong['abs_string_diff'].value_counts().sort_index().items():
        lab = {1:'ADJACENT (one string off)', 2:'two strings off', 3:'three strings off'}.get(k, f'{k} off')
        print(f"  |Δstring|={k}: {v:5d}  ({v/nw:5.1%})  {lab}")
    adj = (wrong['abs_string_diff']==1).mean()
    print(f"\n>>> {adj:.1%} of wrong notes are exactly ONE string off (adjacent).")
    print(f">>> That's {(wrong['abs_string_diff']==1).sum()/n:.1%} of all matched notes "
          f"(the rest place correctly or miss by 2+ strings).")
    print("\nFret-diff distribution (sanity: adjacent swaps cluster at ±4/±5):")
    print(wrong['fret_diff'].value_counts().sort_index().to_string())
    return wrong

# wrong = string_error_audit(audio_matches_df, method='caged_box')

In [34]:
wrong = string_error_audit(audio_matches_df, method='caged_box')


Matched notes (detected + pitch-correct): 7668
  exact-correct placement : 5256  (68.5%)
  wrong placement         : 2412  (31.5%)

Of the WRONG notes — how many strings off:
  |Δstring|=1:  2309  (95.7%)  ADJACENT (one string off)
  |Δstring|=2:    88  ( 3.6%)  two strings off
  |Δstring|=3:    14  ( 0.6%)  three strings off
  |Δstring|=5:     1  ( 0.0%)  5 off

>>> 95.7% of wrong notes are exactly ONE string off (adjacent).
>>> That's 30.1% of all matched notes (the rest place correctly or miss by 2+ strings).

Fret-diff distribution (sanity: adjacent swaps cluster at ±4/±5):
fret_diff
-10       4
-9       20
-5     1190
-4      400
 4      246
 5      473
 9       53
 10      11
 14       6
 15       8
 24       1


In [35]:
import numpy as np
adj = wrong[wrong['abs_string_diff'] == 1].copy()
names = ['E','A','D','G','B','e']  # string index 0..5
adj['kind'] = np.where(adj['recording'].str.endswith('_solo'), 'solo',
              np.where(adj['recording'].str.endswith('_comp'), 'comp', 'other'))
adj['pair'] = adj.apply(lambda r: f"{names[min(r.true_string, r.pred_string)]}–{names[max(r.true_string, r.pred_string)]}", axis=1)
adj['direction'] = np.where(adj['string_diff'] > 0, 'to thinner/higher string', 'to thicker/lower string')

print("Adjacent errors — solo vs comp:")
print(adj['kind'].value_counts(), "\n")
print("Which string pair gets confused:")
print(adj['pair'].value_counts(), "\n")
print("Direction:")
print(adj['direction'].value_counts(normalize=True).round(3))

Adjacent errors — solo vs comp:
kind
comp    1438
solo     871
Name: count, dtype: int64 

Which string pair gets confused:
pair
G–B    646
D–G    605
B–e    467
A–D    373
E–A    218
Name: count, dtype: int64 

Direction:
direction
to thinner/higher string    0.689
to thicker/lower string     0.311
Name: proportion, dtype: float64


In [36]:
import numpy as np
def method_breakdown(df):
    d = df.dropna(subset=['pred_string','true_string']).copy()
    d['kind'] = np.where(d.recording.str.endswith('_solo'),'solo',
                np.where(d.recording.str.endswith('_comp'),'comp','other'))
    rows = []
    for meth, g in d.groupby('method'):
        for sl in ['ALL','comp','solo']:
            gg = g if sl=='ALL' else g[g['kind']==sl]
            if len(gg): rows.append({'method':meth,'slice':sl,
                                     'matched':len(gg),'exact_tab_acc':round(gg['exact_tab_correct'].mean(),4)})
    return pd.DataFrame(rows).pivot(index='method',columns='slice',values='exact_tab_acc')[['ALL','comp','solo']]

display(method_breakdown(audio_matches_df))

slice,ALL,comp,solo
method,,,
caged_box,0.6854,0.7199,0.6105
caged_box_repaired,0.6854,0.7199,0.6105
caged_voiced,0.6918,0.7308,0.6072
caged_voiced_repaired,0.6918,0.7308,0.6072
combined_all_tuned,0.6472,0.6677,0.6026
combined_all_tuned_repaired,0.6472,0.6677,0.6026
